## Layer-wise Relevance Propagation

### Imports

In [ ]:
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from scipy.stats import spearmanr
from scipy.stats import entropy as scipy_entropy

import torch
import torch.nn as nn
import torch.nn.init as init

from captum.attr import LRP, LayerLRP

### build 30_features csv


In [ ]:
CSV_PATH = "Dataset/CSV/paired_data_datscan.csv"

MEDICAL_TOP30_FEATURES = [
    'NP3RISNG', 'NP3FTAPR', 'NP3RIGLU', 'NP3TOT',   'DRMREMEM',
    'NP3RIGN',  'SCAU13',   'SCAU4',    'NP2PTOT',  'PARKISM',
    'ESS5',     'NP2SALV',  'HRPOSTMED','SCAU8',    'SCAU23A',
    'PDTRTMNT', 'NP3RIGRU', 'SCAU26D',  'SCAU26A',  'DRMVERBL',
    'SCAU22',   'NP2EAT',   'SCAU12',   'DRMVIVID', 'DRMNOCTB',
    'SLPDSTRB', 'SCAU26C',  'ESS4',     'NP1COG',   'SCAU14'
]

# Columns to keep
LABEL_COLUMN = "NHY"
ID_COLUMNS = ["PATNO", "EVENT_ID", "Image Data ID"]
selected_columns = ID_COLUMNS + MEDICAL_TOP30_FEATURES + [LABEL_COLUMN]

df = pd.read_csv(CSV_PATH)

df_filtered = df[selected_columns]

output_path = "Dataset/CSV/paired_data_datscan_30_features.csv"
df_filtered.to_csv(output_path, index=False)

print("Saved:", output_path)

### Config + Model + Data

In [ ]:
# CONFIG
MODEL_PATH = "models/MedicalRecordProcessor_WEIGHTS_v7.pth"
CSV_PATH = "Dataset/CSV/paired_data_datscan_30_features.csv"

LABEL_COLUMN = "NHY"
ID_COLUMNS = ["PATNO", "EVENT_ID", "Image Data ID"]

TOP_N_GLOBAL = 10    # number of top global features to plot
TOP_N_PER_CLASS = 7  # number of top features per Parkinson stage
TOP_N_FEATURES = 5   # number of top features to include in stage_progression

OUTPUT_FOLDER = "lrp_analysis_30_thesis"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)



# MODEL DEFINITION; 
# always set classify=True
class MedicalRecordProcessor(nn.Module):
    """
    30 -> 128 -> 256 -> 5 (~40K params)
    Monotonic expansion — no bottleneck.
    """

    def __init__(self, num_features=30, num_classes=5):
        super().__init__()
        self.num_features = num_features
        self.num_classes = num_classes

        self.mlp = nn.Sequential(
            nn.Linear(num_features, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),

            nn.Linear(128, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
        )

        self.classifier = nn.Linear(256, num_classes)

        self._initialize_weights()

    def _initialize_weights(self):
        """He initialization for all linear layers."""
        for m in self.modules():
            if isinstance(m, nn.Linear):
                init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def forward(self, x, classify=True):
        if x.shape[1] != self.num_features:
            raise ValueError(f"Expected {self.num_features} features, got {x.shape[1]}")

        features = self.mlp(x)

        if classify:
            return self.classifier(features)
        return features



# LOAD MODEL
device = torch.device("cpu")
model = MedicalRecordProcessor()
checkpoint = torch.load(MODEL_PATH, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()



# LOAD DATA
df = pd.read_csv(CSV_PATH)

labels = df[LABEL_COLUMN].values.astype(int)

id_df = df[ID_COLUMNS]  # keep IDs to save later

features_df = df.drop(columns=[LABEL_COLUMN] + ID_COLUMNS, errors='ignore')
feature_names = features_df.columns.tolist()
X = features_df.values.astype(np.float32)
X_tensor = torch.tensor(X, dtype=torch.float32)



# helper for plotting
def plot_top_features(df_sorted, n,save_path, title=""):
    top_df = df_sorted.head(n)
    plt.figure(figsize=(8, 5))
    plt.barh(top_df["feature"], top_df["mean_abs_relevance"])
    plt.gca().invert_yaxis()
    plt.title(title)
    plt.xlabel("Mean Absolute Relevance")
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

### LRP implementation

##### with BatchNorm1d layer (v2; changed the 4 new plots)

In [ ]:
# ══════════════════════════════════════════════════════════════
# LRP WRAPPER (removes BatchNorm for compatibility)
# ══════════════════════════════════════════════════════════════
class LRPMLPWrapper(nn.Module):
    def __init__(self, mlp, classifier):
        super().__init__()
        self.layers = nn.Sequential(*(l for l in mlp if not isinstance(l, nn.BatchNorm1d)))
        self.classifier = classifier

    def forward(self, x):
        return self.classifier(self.layers(x))

lrp_model = LRPMLPWrapper(model.mlp, model.classifier)
lrp_model.eval()

layers_to_explain = [
    ("layer1_linear", lrp_model.layers[0]),
    ("layer2_linear", lrp_model.layers[3]),
    ("classifier",    lrp_model.classifier),
]
layer_lrps    = {name: LayerLRP(lrp_model, layer) for name, layer in layers_to_explain}
layer_outputs = {name: [] for name, _ in layers_to_explain}
lrp           = LRP(lrp_model)


# ══════════════════════════════════════════════════════════════
# LRP COMPUTATION
# ══════════════════════════════════════════════════════════════
all_relevance = []
pred_classes  = []

for i in range(X_tensor.shape[0]):
    x          = X_tensor[i].unsqueeze(0)
    pred_class = lrp_model(x).argmax(dim=1).item()
    pred_classes.append(pred_class)

    relevance = lrp.attribute(x, target=pred_class).detach().cpu().numpy().flatten()
    all_relevance.append(relevance)

    for layer_name, layer_lrp in layer_lrps.items():
        layer_rel = layer_lrp.attribute(x, target=pred_class).detach().cpu().numpy().flatten()
        layer_outputs[layer_name].append(layer_rel)

all_relevance = np.array(all_relevance)
pred_classes  = np.array(pred_classes)
correct_mask  = pred_classes == labels
unique_stages = sorted(np.unique(labels))
n_stages      = len(unique_stages)

global_mean_signed = all_relevance.mean(axis=0)
global_mean_abs    = np.abs(all_relevance).mean(axis=0)

global_df = pd.DataFrame({
    "feature":               feature_names,
    "mean_signed_relevance": global_mean_signed,
    "mean_abs_relevance":    global_mean_abs,
}).sort_values("mean_abs_relevance", ascending=False)

stage_cmap   = plt.colormaps['tab10'].resampled(n_stages)
stage_colors = {s: stage_cmap(i) for i, s in enumerate(unique_stages)}

print(f"Accuracy: {correct_mask.mean():.3f}  "
      f"({correct_mask.sum()} correct / {(~correct_mask).sum()} misclassified)")


# ══════════════════════════════════════════════════════════════
# LAYER RELEVANCE
# ══════════════════════════════════════════════════════════════
# layer_folder = os.path.join(OUTPUT_FOLDER, "layer_relevance")
# os.makedirs(layer_folder, exist_ok=True)

# for layer_name, values in layer_outputs.items():
#     layer_array = np.array(values)
#     cur_folder  = os.path.join(layer_folder, layer_name)
#     os.makedirs(cur_folder, exist_ok=True)

#     layer_df = pd.DataFrame(layer_array)
#     layer_df = pd.concat([id_df.reset_index(drop=True), layer_df], axis=1)
#     layer_df.to_csv(os.path.join(cur_folder, "relevance_per_patient.csv"), index=False)

#     median_relevance = np.median(layer_array, axis=0)
#     pd.DataFrame({
#         "neuron_index":     range(len(median_relevance)),
#         "median_relevance": median_relevance,
#     }).to_csv(os.path.join(cur_folder, "median_relevance.csv"), index=False)

#     neuron_means = np.abs(layer_array).mean(axis=0)
#     n_show       = min(20, len(median_relevance))

#     # median signed bar
#     top_idx_m  = np.argsort(np.abs(median_relevance))[-n_show:]
#     top_vals   = median_relevance[top_idx_m]
#     fig, ax = plt.subplots(figsize=(9, 6))
#     ax.barh([f"N{i}" for i in top_idx_m], top_vals,
#             color=["#C44E52" if v > 0 else "#4C72B0" for v in top_vals],
#             edgecolor="white", linewidth=0.4)
#     ax.axvline(0, color="black", linewidth=0.8)
#     ax.set_xlabel("Median Signed LRP Relevance", fontsize=11)
#     ax.set_title(f"{layer_name} — Top {n_show} Neurons by Median Relevance\n"
#                  "(red = promotes prediction | blue = suppresses)", fontsize=11, fontweight="bold")
#     ax.invert_yaxis()
#     ax.grid(True, axis='x', linestyle='--', alpha=0.4)
#     plt.tight_layout()
#     plt.savefig(os.path.join(cur_folder, "median_relevance_bar.png"), dpi=180)
#     plt.close()

#     # neuron relevance violin
#     fig, ax = plt.subplots(figsize=(7, 5))
#     parts = ax.violinplot(neuron_means, positions=[0], showmedians=True, showextrema=True)
#     for pc in parts['bodies']:
#         pc.set_facecolor("#4C72B0"); pc.set_alpha(0.7)
#     parts['cmedians'].set_color("#C44E52"); parts['cmedians'].set_linewidth(2)
#     ax.set_xticks([0])
#     ax.set_xticklabels([layer_name.replace("_", " ")], fontsize=10)
#     ax.set_ylabel("Mean Abs. Neuron Relevance", fontsize=10)
#     ax.set_title(f"{layer_name} — Neuron Relevance Distribution\n"
#                  f"({layer_array.shape[1]} neurons × {layer_array.shape[0]} patients)",
#                  fontsize=11, fontweight="bold")
#     ax.grid(True, linestyle='--', alpha=0.4)
#     plt.tight_layout()
#     plt.savefig(os.path.join(cur_folder, "neuron_relevance_violin.png"), dpi=180)
#     plt.close()

#     # top neuron heatmap per stage
#     n_top_h     = min(20, layer_array.shape[1])
#     top_h_idx   = np.argsort(np.abs(neuron_means))[-n_top_h:]
#     stage_mat   = np.zeros((n_stages, n_top_h))
#     for si, stage in enumerate(unique_stages):
#         stage_mat[si] = np.abs(layer_array[labels == stage])[:, top_h_idx].mean(axis=0)

#     fig, ax = plt.subplots(figsize=(14, 4))
#     im = ax.imshow(stage_mat, aspect='auto', cmap='YlOrRd', interpolation='nearest')
#     plt.colorbar(im, ax=ax, label="Mean Abs. Relevance")
#     ax.set_yticks(range(n_stages))
#     ax.set_yticklabels([f"Stage {s}" for s in unique_stages], fontsize=10)
#     ax.set_xticks(range(n_top_h))
#     ax.set_xticklabels([f"N{i}" for i in top_h_idx], rotation=45, ha='right', fontsize=8)
#     ax.set_title(f"{layer_name} — Top {n_top_h} Neuron Relevance per Parkinson Stage",
#                  fontsize=11, fontweight="bold")
#     plt.tight_layout()
#     plt.savefig(os.path.join(cur_folder, "neuron_heatmap_per_stage.png"), dpi=180)
#     plt.close()


# ══════════════════════════════════════════════════════════════
# RELEVANCE FLOW ACROSS LAYERS
# ══════════════════════════════════════════════════════════════
# flow_folder = os.path.join(OUTPUT_FOLDER, "relevance_flow")
# os.makedirs(flow_folder, exist_ok=True)

# flow_layers = ["Input\nFeatures"] + \
#               [ln.replace("_", "\n") for ln in layer_outputs] + \
#               ["Prediction"]
# flow_values = [np.mean(np.abs(all_relevance))] + \
#               [np.mean(np.abs(np.array(v))) for v in layer_outputs.values()] + \
#               [1.0]

# fig, ax = plt.subplots(figsize=(10, 5))
# x_pos = range(len(flow_layers))
# ax.plot(x_pos, flow_values, marker='o', linewidth=2, color="#4C72B0", markersize=9, zorder=3)
# ax.fill_between(x_pos, flow_values, alpha=0.15, color="#4C72B0")
# for xp, yp in zip(x_pos, flow_values):
#     ax.annotate(f"{yp:.4f}", (xp, yp), textcoords="offset points",
#                 xytext=(0, 12), ha='center', fontsize=9)
# ax.set_xticks(x_pos)
# ax.set_xticklabels(flow_layers, fontsize=10)
# ax.set_ylabel("Mean Absolute Relevance", fontsize=11)
# ax.set_xlabel("Network Stage", fontsize=11)
# ax.grid(True, linestyle='--', alpha=0.4)
# plt.tight_layout()
# plt.savefig(os.path.join(flow_folder, "relevance_flow_across_layers.png"), dpi=180)
# plt.close()

# pd.DataFrame({"layer": flow_layers, "mean_absolute_relevance": flow_values}).to_csv(
#     os.path.join(flow_folder, "relevance_flow_across_layers.csv"), index=False)


# ══════════════════════════════════════════════════════════════
# PER PATIENT RELEVANCE — CSV
# ══════════════════════════════════════════════════════════════
# per_patient_df = pd.DataFrame(all_relevance, columns=feature_names)
# per_patient_df = pd.concat([id_df.reset_index(drop=True), per_patient_df], axis=1)
# per_patient_df[LABEL_COLUMN]      = labels
# per_patient_df["predicted_class"] = pred_classes
# per_patient_df["correct"]         = correct_mask
# per_patient_df.to_csv(f"{OUTPUT_FOLDER}/relevance_per_patient.csv", index=False)


# ══════════════════════════════════════════════════════════════
# GLOBAL IMPORTANCE
# ══════════════════════════════════════════════════════════════
# global_folder = os.path.join(OUTPUT_FOLDER, "global_importance")
# os.makedirs(global_folder, exist_ok=True)
# global_df.to_csv(f"{global_folder}/global_feature_importance.csv", index=False)
 
# plot_top_features(global_df, TOP_N_GLOBAL, f"{global_folder}/top_global_features.png")
 
# top_g    = global_df.head(TOP_N_GLOBAL).copy()
# fig, ax  = plt.subplots(figsize=(9, 6))
# ax.barh(top_g["feature"], top_g["mean_signed_relevance"],
#         color=["#C44E52" if v > 0 else "#4C72B0" for v in top_g["mean_signed_relevance"]],
#         edgecolor="white", linewidth=0.5)
# ax.axvline(0, color="black", linewidth=0.8)
# ax.set_xlabel("Mean Signed Relevance", fontsize=11)
# ax.set_title(f"Top {TOP_N_GLOBAL} Global Features — Signed Relevance\n"
#              "(red = promotes class | blue = suppresses class)", fontsize=12, fontweight="bold")
# ax.invert_yaxis()
# ax.grid(True, axis='x', linestyle='--', alpha=0.4)
# plt.tight_layout()
# plt.savefig(f"{global_folder}/top_global_features_signed.png", dpi=180)
# plt.close()
 
# sorted_abs = global_df.sort_values("mean_abs_relevance", ascending=True)
# vmax_signed = sorted_abs["mean_signed_relevance"].abs().max()
# fig, ax = plt.subplots(figsize=(8, max(6, len(feature_names) * 0.22)))
# sc = ax.scatter(sorted_abs["mean_abs_relevance"], sorted_abs["feature"],
#                 c=sorted_abs["mean_signed_relevance"], cmap="RdBu_r", s=60,
#                 edgecolors="grey", linewidths=0.4,
#                 norm=plt.Normalize(vmin=-vmax_signed, vmax=vmax_signed))
# plt.colorbar(plt.cm.ScalarMappable(cmap="RdBu_r",
#              norm=plt.Normalize(vmin=-vmax_signed, vmax=vmax_signed)),
#              ax=ax, label="Mean Signed Relevance")
# ax.set_xlabel("Mean Absolute Relevance", fontsize=11)
# ax.set_xlim(left=0)
# ax.set_title("All Features - Global LRP Relevance\n"
#              "(dot color = signed direction | position = magnitude)",
#              fontsize=11, fontweight="bold")
# ax.grid(True, axis='x', linestyle='--', alpha=0.3)
# plt.tight_layout()
# plt.savefig(f"{global_folder}/all_features_dot_plot.png", dpi=180)
# plt.close()


# ══════════════════════════════════════════════════════════════
# PER CLASS IMPORTANCE
# ══════════════════════════════════════════════════════════════
class_importance = {}
per_class_folder = os.path.join(OUTPUT_FOLDER, "class_importance")
os.makedirs(per_class_folder, exist_ok=True)

for c in sorted(np.unique(labels)):
    mask     = labels == c
    rel      = all_relevance[mask]
    df_class = pd.DataFrame({
        "feature":               feature_names,
        "mean_signed_relevance": rel.mean(axis=0),
        "mean_abs_relevance":    np.abs(rel).mean(axis=0),
        "std_relevance":         np.abs(rel).std(axis=0),
    }).sort_values("mean_abs_relevance", ascending=False)
    df_class.to_csv(f"{per_class_folder}/class_{c}_feature_importance.csv", index=False)
    class_importance[c] = df_class

    plot_top_features(df_class, TOP_N_PER_CLASS,
                      f"{per_class_folder}/top_features_class_{c}.png")

    top_c   = df_class.head(TOP_N_PER_CLASS).copy()
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.barh(top_c["feature"], top_c["mean_signed_relevance"],
            xerr=top_c["std_relevance"],
            color=["#C44E52" if v > 0 else "#4C72B0" for v in top_c["mean_signed_relevance"]],
            edgecolor="white", linewidth=0.5,
            error_kw=dict(elinewidth=0.8, ecolor="gray", capsize=3))
    ax.axvline(0, color="black", linewidth=0.8)
    ax.invert_yaxis()
    ax.set_xlabel("Mean Signed LRP Relevance (± std)", fontsize=11)
    ax.set_title(f"Stage {c} — Top {TOP_N_PER_CLASS} Features  (n={mask.sum()} patients)",
                 fontsize=11, fontweight="bold")
    ax.grid(True, axis='x', linestyle='--', alpha=0.4)
    plt.tight_layout()
    plt.savefig(f"{per_class_folder}/class_{c}_signed_bar.png", dpi=180)
    plt.close()

fig, axes = plt.subplots(1, n_stages, figsize=(5 * n_stages, 7), sharey=False)
if n_stages == 1: axes = [axes]
for ax, stage in zip(axes, unique_stages):
    top_s = class_importance[stage].head(TOP_N_PER_CLASS)
    ax.barh(top_s["feature"], top_s["mean_signed_relevance"],
            color=["#C44E52" if v > 0 else "#4C72B0" for v in top_s["mean_signed_relevance"]],
            edgecolor='white', linewidth=0.4)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(f"Stage {stage}  (n={(labels==stage).sum()})",
                 fontsize=11, fontweight="bold", color=stage_colors[stage])
    ax.set_xlabel("Mean Signed Relevance", fontsize=9)
    ax.invert_yaxis()
    ax.tick_params(axis='y', labelsize=8)
plt.suptitle(f"Top {TOP_N_PER_CLASS} Feature Relevance Per Parkinson Stage (Signed)",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(per_class_folder, "all_stages_signed_small_multiples.png"), dpi=180)
plt.close()


# ══════════════════════════════════════════════════════════════
# STAGE PROGRESSION IMPORTANCE CURVE
# ══════════════════════════════════════════════════════════════
# stage_feature_importance = np.zeros((n_stages, len(feature_names)))
# for i, stage in enumerate(unique_stages):
#     stage_feature_importance[i] = np.abs(all_relevance[labels == stage]).mean(axis=0)

# top_features_indices         = np.argsort(global_mean_abs)[-TOP_N_FEATURES:]
# top_feature_names            = [feature_names[i] for i in top_features_indices]
# stage_feature_importance_top = stage_feature_importance[:, top_features_indices]

# fig, ax = plt.subplots(figsize=(12, 6))
# cmap_prog = plt.colormaps['tab10'].resampled(len(top_feature_names))
# for i, feat_name in enumerate(top_feature_names):
#     ax.plot(unique_stages, stage_feature_importance_top[:, i],
#             marker='o', label=feat_name, color=cmap_prog(i), linewidth=2)
# ax.set_xticks(unique_stages)
# ax.set_xticklabels([f"Stage {s}" for s in unique_stages], fontsize=11)
# ax.set_xlabel("Parkinson Stage (H&Y)", fontsize=11)
# ax.set_ylabel("Mean Absolute LRP Relevance", fontsize=11)
# ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
# ax.grid(True, linestyle='--', alpha=0.4)
# plt.tight_layout()
# plt.savefig(os.path.join(OUTPUT_FOLDER, "stage_progression_importance_curve.png"), dpi=180)
# plt.close()


# ══════════════════════════════════════════════════════════════
# CORRECT vs MISCLASSIFIED
# ══════════════════════════════════════════════════════════════
# correct_folder = os.path.join(OUTPUT_FOLDER, "correct_vs_misclassified")
# os.makedirs(correct_folder, exist_ok=True)
 
# correct_rel   = np.abs(all_relevance[correct_mask]).mean(axis=0)
# incorrect_rel = np.abs(all_relevance[~correct_mask]).mean(axis=0)
# diff_rel      = correct_rel - incorrect_rel
# top_diff_idx  = np.argsort(np.abs(diff_rel))[-TOP_N_GLOBAL:]
# top_diff_names = [feature_names[i] for i in top_diff_idx]
 
# fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
# axes[0].barh(top_diff_names,  correct_rel[top_diff_idx],   color="#55A868", label="Correct")
# axes[0].barh(top_diff_names, -incorrect_rel[top_diff_idx], color="#C44E52", label="Misclassified")
# axes[0].axvline(0, color='black', linewidth=0.8)
# axes[0].set_xlabel("Mean Absolute Relevance")
# axes[0].set_title("Feature Relevance: Correct vs Misclassified", fontweight="bold")
# axes[0].invert_yaxis()
# axes[1].barh(top_diff_names, diff_rel[top_diff_idx],
#              color=["#55A868" if d > 0 else "#C44E52" for d in diff_rel[top_diff_idx]])
# axes[1].axvline(0, color='black', linewidth=0.8)
# axes[1].set_xlabel("Δ Relevance (Correct – Misclassified)")
# axes[1].set_title("Relevance Difference", fontweight="bold")
# plt.suptitle(f"LRP Relevance: Correctly vs Incorrectly Classified Patients\n"
#              f"(n_correct={correct_mask.sum()}, n_misclassified={(~correct_mask).sum()})",
#              fontsize=12, fontweight="bold")
# fig.legend(handles=[
#     mpatches.Patch(color="#55A868", label="Correct"),
#     mpatches.Patch(color="#C44E52", label="Misclassified"),
# ], loc='lower center', ncol=2, fontsize=10, bbox_to_anchor=(0.5, -0.04))
# plt.tight_layout()
# plt.savefig(os.path.join(correct_folder, "correct_vs_misclassified_relevance.png"),
#             dpi=180, bbox_inches='tight')
# plt.close()
 
# pd.DataFrame({
#     "feature": feature_names, "correct_rel": correct_rel,
#     "incorrect_rel": incorrect_rel, "delta": diff_rel,
# }).sort_values("delta", ascending=False).to_csv(
#     os.path.join(correct_folder, "correct_vs_misclassified_relevance.csv"), index=False)


# ══════════════════════════════════════════════════════════════
# VARIANCE vs RELEVANCE
# ══════════════════════════════════════════════════════════════
# relevance_rank = np.argsort(np.argsort(global_mean_abs))
# feat_variance  = X.var(axis=0)

# fig, ax = plt.subplots(figsize=(8, 6))
# sc = ax.scatter(feat_variance, global_mean_abs, alpha=0.6, s=40,
#                 c=relevance_rank, cmap="viridis", edgecolors="none")
# for idx in np.argsort(global_mean_abs)[-5:]:
#     ax.annotate(feature_names[idx], (feat_variance[idx], global_mean_abs[idx]),
#                 fontsize=7, xytext=(4, 4), textcoords="offset points")
# plt.colorbar(sc, ax=ax, label="Relevance rank")
# ax.set_xlabel("Feature Variance (in dataset)", fontsize=11)
# ax.set_ylabel("Mean Absolute LRP Relevance", fontsize=11)
# ax.set_title("Feature Variance vs LRP Relevance\n"
#              "(Do high-variance features get more credit?)", fontsize=12, fontweight="bold")
# rho, p = spearmanr(feat_variance, global_mean_abs)
# ax.text(0.05, 0.92, f"Spearman ρ = {rho:.3f}  (p={p:.3f})", transform=ax.transAxes,
#         fontsize=10, bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray"))
# plt.tight_layout()
# plt.savefig(os.path.join(OUTPUT_FOLDER, "variance_vs_relevance.png"), dpi=180)
# plt.close()


# ══════════════════════════════════════════════════════════════
# LAYER SPECIALIZATION ENTROPY
# ══════════════════════════════════════════════════════════════
# layer_arrays = {name: np.array(vals) for name, vals in layer_outputs.items()}
# layer_names  = list(layer_arrays.keys())

# entropy_folder = os.path.join(OUTPUT_FOLDER, "layer_relevance", "plots")
# os.makedirs(entropy_folder, exist_ok=True)

# entropy_per_stage = {name: [] for name in layer_names}
# for name, arr in layer_arrays.items():
#     for stage in unique_stages:
#         stage_mean = np.abs(arr[labels == stage]).mean(axis=0)
#         prob       = stage_mean / (stage_mean.sum() + 1e-8)
#         entropy_per_stage[name].append(scipy_entropy(prob + 1e-10))

# x           = np.arange(n_stages)
# width       = 0.8 / len(layer_names)
# cmap_layers = plt.colormaps['Set2'].resampled(len(layer_names))

# fig, ax = plt.subplots(figsize=(9, 5))
# for i, name in enumerate(layer_names):
#     offset = (i - len(layer_names) / 2 + 0.5) * width
#     ax.bar(x + offset, entropy_per_stage[name], width=width, linewidth=0.5,
#            label=name.replace("_", " "), color=cmap_layers(i), edgecolor='white')
# ax.set_xticks(x)
# ax.set_xticklabels([f"Stage {s}" for s in unique_stages], fontsize=11)
# ax.set_xlabel("Parkinson Stage", fontsize=11)
# ax.set_ylabel("Relevance Entropy", fontsize=11)
# ax.set_title("Layer Specialization — Relevance Entropy per Layer per Stage\n"
#              "(Low entropy = few neurons dominate | High entropy = distributed)",
#              fontsize=12, fontweight="bold")
# ax.legend(fontsize=9)
# ax.grid(True, axis='y', linestyle='--', alpha=0.4)
# plt.tight_layout()
# plt.savefig(os.path.join(entropy_folder, "layer_specialization_entropy.png"), dpi=180)
# plt.close()

# entropy_df = pd.DataFrame(entropy_per_stage, index=[f"Stage {s}" for s in unique_stages])
# entropy_df.index.name = "stage"
# entropy_df.columns = [c.replace("_", " ") for c in entropy_df.columns]
# entropy_df.to_csv(os.path.join(entropy_folder, "layer_specialization_entropy.csv"))


# ══════════════════════════════════════════════════════════════
# CROSS-LAYER RELEVANCE PROPAGATION
# ══════════════════════════════════════════════════════════════
# fig, ax = plt.subplots(figsize=(10, 5))
# cmap_l = plt.colormaps['tab10'].resampled(len(layer_names))
# for i, (name, arr) in enumerate(layer_arrays.items()):
#     ax.plot(unique_stages, [np.abs(arr[labels == s]).mean() for s in unique_stages],
#             marker='o', linewidth=2, label=name.replace("_", " "), color=cmap_l(i))
# ax.set_xticks(unique_stages)
# ax.set_xticklabels([f"Stage {s}" for s in unique_stages], fontsize=11)
# ax.set_xlabel("Parkinson Stage", fontsize=11)
# ax.set_ylabel("Mean Absolute Relevance", fontsize=11)
# ax.set_title("Cross-Layer Relevance Propagation Across Disease Stages\n"
#              "(How much relevance each layer carries per stage)",
#              fontsize=12, fontweight="bold")
# ax.legend(title="Layer", fontsize=9)
# ax.grid(True, linestyle='--', alpha=0.4)
# plt.tight_layout()
# plt.savefig(os.path.join(entropy_folder, "cross_layer_relevance_propagation.png"), dpi=180)
# plt.close()


# ══════════════════════════════════════════════════════════════
# EXTRA PLOTS  (built from class_X_feature_importance.csv files)
# ══════════════════════════════════════════════════════════════
TOP_N_STAGE   = 8    # features per stage for graphs 1 and 4
MIN_RELEVANCE = 0.8  # threshold for graph 2 journey lines
 
class_dfs = {
    c: pd.read_csv(os.path.join(OUTPUT_FOLDER, "class_importance",
                                f"class_{c}_feature_importance.csv"))
    for c in unique_stages
}
 
 
# ── Graph 1: Top Features Per Stage (absolute) ────────────────
# g1_folder = os.path.join(OUTPUT_FOLDER, "top_features_per_stage")
# os.makedirs(g1_folder, exist_ok=True)
 
# fig, axes = plt.subplots(1, n_stages, figsize=(5 * n_stages, 7), sharey=False)
# if n_stages == 1: axes = [axes]
 
# g1_rows = []
# for ax, stage in zip(axes, unique_stages):
#     top = class_dfs[stage].nlargest(TOP_N_STAGE, "mean_abs_relevance")
#     ax.barh(top["feature"], top["mean_abs_relevance"],
#             color=stage_colors[stage], edgecolor="white", linewidth=0.5)
#     ax.invert_yaxis()
#     ax.set_title(f"Stage {stage}  (n={(labels==stage).sum()})",
#                  fontsize=11, fontweight="bold", color=stage_colors[stage])
#     ax.set_xlabel("Mean Abs. Relevance", fontsize=9)
#     ax.tick_params(axis='y', labelsize=8)
#     ax.grid(True, axis='x', linestyle='--', alpha=0.4)
#     for _, row in top.iterrows():
#         g1_rows.append({"stage": stage, "feature": row["feature"],
#                         "mean_abs_relevance": row["mean_abs_relevance"],
#                         "rank": list(top["feature"]).index(row["feature"]) + 1})
 
# plt.suptitle(f"Top {TOP_N_STAGE} Most Important Features per Stage",
#              fontsize=14, fontweight="bold")
# plt.tight_layout()
# plt.savefig(os.path.join(g1_folder, "top_features_per_stage.png"),
#             dpi=180, bbox_inches='tight')
# plt.close()
 
# pd.DataFrame(g1_rows).to_csv(os.path.join(g1_folder, "top_features_per_stage.csv"), index=False)
 
 
# ── Graph 2: Feature Journey Across Stages ────────────────────
# g2_folder = os.path.join(OUTPUT_FOLDER, "feature_journey_across_stages")
# os.makedirs(g2_folder, exist_ok=True)
 
# all_top_features = set()
# for df in class_dfs.values():
#     all_top_features.update(df.nlargest(TOP_N_STAGE, "mean_abs_relevance")["feature"].tolist())
 
# journey = {}
# for feat in all_top_features:
#     scores = []
#     for stage in unique_stages:
#         row = class_dfs[stage][class_dfs[stage]["feature"] == feat]
#         scores.append(float(row["mean_abs_relevance"].values[0]) if len(row) else 0.0)
#     if max(scores) >= MIN_RELEVANCE:
#         journey[feat] = scores
 
# fig, ax = plt.subplots(figsize=(12, 6))
# cmap_j = plt.colormaps['tab10'].resampled(len(journey))
 
# for i, (feat, scores) in enumerate(sorted(journey.items(),
#                                            key=lambda x: max(x[1]), reverse=True)):
#     ax.plot(unique_stages, scores, marker='o', linewidth=2,
#             label=feat, color=cmap_j(i))
 
# ax.set_xticks(unique_stages)
# ax.set_xticklabels([f"Stage {s}" for s in unique_stages], fontsize=11)
# ax.set_xlabel("Parkinson Stage", fontsize=11)
# ax.set_ylabel("Mean Absolute LRP Relevance", fontsize=11)
# ax.set_title(f"Feature Journey Across Stages\n"
#              f"(features reaching relevance ≥ {MIN_RELEVANCE} in at least one stage)",
#              fontsize=12, fontweight="bold")
# ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
# ax.grid(True, linestyle='--', alpha=0.4)
# plt.tight_layout()
# plt.savefig(os.path.join(g2_folder, "feature_journey_across_stages.png"),
#             dpi=180, bbox_inches='tight')
# plt.close()
 
# journey_df = pd.DataFrame(journey, index=[f"Stage {s}" for s in unique_stages]).T
# journey_df.index.name = "feature"
# journey_df.columns.name = None
# journey_df["peak_stage"]     = journey_df.idxmax(axis=1)
# journey_df["peak_relevance"] = journey_df.drop(columns="peak_stage").max(axis=1)
# journey_df = journey_df.sort_values("peak_relevance", ascending=False)
# journey_df.to_csv(os.path.join(g2_folder, "feature_journey_across_stages.csv"))
 
 
# ── Graph 3: Feature vs Stage Heatmap (column-normalized) ─────
g3_folder = os.path.join(OUTPUT_FOLDER, "feature_stage_heatmap")
os.makedirs(g3_folder, exist_ok=True)
 
top10_per_stage = set()
for df in class_dfs.values():
    top10_per_stage.update(df.nlargest(10, "mean_abs_relevance")["feature"].tolist())
top10_per_stage = sorted(top10_per_stage)
 
heatmap_data = np.zeros((n_stages, len(top10_per_stage)))
for si, stage in enumerate(unique_stages):
    for fi, feat in enumerate(top10_per_stage):
        row = class_dfs[stage][class_dfs[stage]["feature"] == feat]
        heatmap_data[si, fi] = float(row["mean_abs_relevance"].values[0]) if len(row) else 0.0
 
col_max = heatmap_data.max(axis=0)
col_max[col_max == 0] = 1
heatmap_norm = heatmap_data / col_max
 
fig, ax = plt.subplots(figsize=(max(14, len(top10_per_stage) * 0.7), 5))
im = ax.imshow(heatmap_norm, aspect='auto', cmap='YlOrBr', vmin=0, vmax=1,
               interpolation='nearest')
plt.colorbar(im, ax=ax, label="Normalized Relevance (per feature column)")
ax.set_yticks(range(n_stages))
ax.set_yticklabels([f"Stage {s}" for s in unique_stages], fontsize=11)
ax.set_xticks(range(len(top10_per_stage)))
ax.set_xticklabels(top10_per_stage, rotation=45, ha='right', fontsize=9)
for si in range(n_stages):
    for fi in range(len(top10_per_stage)):
        ax.text(fi, si, f"{heatmap_data[si, fi]:.2f}", ha='center', va='center',
                fontsize=7, color='black' if heatmap_norm[si, fi] < 0.6 else 'white')
ax.set_title("Feature vs Stage Relevance Heatmap\n"
             "(columns normalized — darker = more important for that feature across stages)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(g3_folder, "feature_stage_heatmap.png"),
            dpi=180, bbox_inches='tight')
plt.close()
 
heatmap_raw_df = pd.DataFrame(heatmap_data,
                               index=[f"Stage {s}" for s in unique_stages],
                               columns=top10_per_stage)
heatmap_raw_df.index.name = "stage"
heatmap_norm_df = pd.DataFrame(heatmap_norm,
                                index=[f"Stage {s}" for s in unique_stages],
                                columns=top10_per_stage)
heatmap_norm_df.index.name = "stage"
heatmap_raw_df.to_csv(os.path.join(g3_folder, "feature_stage_heatmap_raw.csv"))
heatmap_norm_df.to_csv(os.path.join(g3_folder, "feature_stage_heatmap_normalized.csv"))
 
 
# ── Graph 4: Does the Feature Push or Pull? (signed) ──────────
g4_folder = os.path.join(OUTPUT_FOLDER, "feature_push_or_pull")
os.makedirs(g4_folder, exist_ok=True)
 
fig = plt.figure(figsize=(18, 11))
grid = fig.add_gridspec(2, 6, hspace=0.2, wspace=0.7)
 
axes = [
    fig.add_subplot(grid[0, 0:2]),
    fig.add_subplot(grid[0, 2:4]),
    fig.add_subplot(grid[0, 4:6]),
    fig.add_subplot(grid[1, 1:3]),
    fig.add_subplot(grid[1, 3:5]),
]
 
g4_rows = []
for ax, stage in zip(axes, unique_stages):
    top = class_dfs[stage].nlargest(TOP_N_STAGE, "mean_abs_relevance")
    colors = ["#55A868" if v > 0 else "#C44E52" for v in top["mean_signed_relevance"]]
    ax.barh(top["feature"], top["mean_signed_relevance"],
            color=colors, edgecolor="white", linewidth=0.3)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.invert_yaxis()
    ax.set_title(f"Stage {stage}  (n={(labels==stage).sum()})",
                 fontsize=11, fontweight="bold", color="black")
    ax.set_xlabel("Mean Signed Relevance", fontsize=9)
    ax.tick_params(axis='y', labelsize=8, pad=2)
    ax.yaxis.set_tick_params(length=0)
    ax.grid(True, axis='x', linestyle='--', alpha=0.4)
    for _, row in top.iterrows():
        g4_rows.append({"stage": stage, "feature": row["feature"],
                        "mean_signed_relevance": row["mean_signed_relevance"],
                        "mean_abs_relevance":    row["mean_abs_relevance"],
                        "direction": "positive" if row["mean_signed_relevance"] > 0 else "negative"})
 
fig.legend(handles=[
    mpatches.Patch(color="#55A868", label="Positive — confirms diagnosis"),
    mpatches.Patch(color="#C44E52", label="Negative — works against diagnosis"),
], loc='lower center', ncol=2, fontsize=10, bbox_to_anchor=(0.5, 0.01))
plt.suptitle(f"Does the Feature Push or Pull? — Top {TOP_N_STAGE} Features per Stage (Signed)",
             fontsize=14, fontweight="bold")
plt.savefig(os.path.join(g4_folder, "feature_push_or_pull_per_stage.png"),
            dpi=180, bbox_inches='tight')
plt.close()
 
pd.DataFrame(g4_rows).to_csv(os.path.join(g4_folder, "feature_push_or_pull_per_stage.csv"),
                              index=False)


print("\nLRP analysis complete. Results saved in:", OUTPUT_FOLDER)

d:\USTHB\M2\Stage Parkinson\Parkinson-prediction\.venv\Lib\site-packages\captum\attr\_core\lrp.py:207: UserWarning: Input Tensor 0 did not already require gradients, required_grads has been set automatically.
  gradient_mask = apply_gradient_requirements(input_tuple)
d:\USTHB\M2\Stage Parkinson\Parkinson-prediction\.venv\Lib\site-packages\captum\attr\_core\layer\layer_lrp.py:233: UserWarning: Input Tensor 0 did not already require gradients, required_grads has been set automatically.
  gradient_mask = apply_gradient_requirements(inputs_tuple)


Accuracy: 0.879  (867 correct / 119 misclassified)

LRP analysis complete. Results saved in: lrp_analysis_30_thesisèfdf


##### with BatchNorm1d layer (v1;added plots to match csv + 4 more plots)

In [18]:
import matplotlib.patches as mpatches
from scipy.spatial.distance import cosine

# ══════════════════════════════════════════════════════════════
# LRP WRAPPER (removes BatchNorm for compatibility)
# ══════════════════════════════════════════════════════════════
class LRPMLPWrapper(nn.Module):
    def __init__(self, mlp, classifier):
        super().__init__()
        self.layers = nn.Sequential(*(l for l in mlp if not isinstance(l, nn.BatchNorm1d)))
        self.classifier = classifier

    def forward(self, x):
        return self.classifier(self.layers(x))

lrp_model = LRPMLPWrapper(model.mlp, model.classifier)
lrp_model.eval()

layers_to_explain = [
    ("layer1_linear", lrp_model.layers[0]),
    ("layer2_linear", lrp_model.layers[3]),
    ("classifier",    lrp_model.classifier),
]
layer_lrps    = {name: LayerLRP(lrp_model, layer) for name, layer in layers_to_explain}
layer_outputs = {name: [] for name, _ in layers_to_explain}
lrp           = LRP(lrp_model)


# ══════════════════════════════════════════════════════════════
# LRP COMPUTATION
# ══════════════════════════════════════════════════════════════
all_relevance = []
pred_classes  = []

for i in range(X_tensor.shape[0]):
    x          = X_tensor[i].unsqueeze(0)
    pred_class = lrp_model(x).argmax(dim=1).item()
    pred_classes.append(pred_class)

    relevance = lrp.attribute(x, target=pred_class).detach().cpu().numpy().flatten()
    all_relevance.append(relevance)

    for layer_name, layer_lrp in layer_lrps.items():
        layer_rel = layer_lrp.attribute(x, target=pred_class).detach().cpu().numpy().flatten()
        layer_outputs[layer_name].append(layer_rel)

all_relevance = np.array(all_relevance)
pred_classes  = np.array(pred_classes)
correct_mask  = pred_classes == labels
unique_stages = sorted(np.unique(labels))
n_stages      = len(unique_stages)

global_mean_signed = all_relevance.mean(axis=0)
global_mean_abs    = np.abs(all_relevance).mean(axis=0)

global_df = pd.DataFrame({
    "feature":               feature_names,
    "mean_signed_relevance": global_mean_signed,
    "mean_abs_relevance":    global_mean_abs,
}).sort_values("mean_abs_relevance", ascending=False)

stage_cmap   = plt.colormaps['tab10'].resampled(n_stages)
stage_colors = {s: stage_cmap(i) for i, s in enumerate(unique_stages)}

print(f"Accuracy: {correct_mask.mean():.3f}  "
      f"({correct_mask.sum()} correct / {(~correct_mask).sum()} misclassified)")


# ══════════════════════════════════════════════════════════════
# LAYER RELEVANCE
# ══════════════════════════════════════════════════════════════
layer_folder = os.path.join(OUTPUT_FOLDER, "layer_relevance")
os.makedirs(layer_folder, exist_ok=True)

for layer_name, values in layer_outputs.items():
    layer_array = np.array(values)
    cur_folder  = os.path.join(layer_folder, layer_name)
    os.makedirs(cur_folder, exist_ok=True)

    layer_df = pd.DataFrame(layer_array)
    layer_df = pd.concat([id_df.reset_index(drop=True), layer_df], axis=1)
    layer_df.to_csv(os.path.join(cur_folder, "relevance_per_patient.csv"), index=False)

    median_relevance = np.median(layer_array, axis=0)
    pd.DataFrame({
        "neuron_index":     range(len(median_relevance)),
        "median_relevance": median_relevance,
    }).to_csv(os.path.join(cur_folder, "median_relevance.csv"), index=False)

    neuron_means = np.abs(layer_array).mean(axis=0)
    n_show       = min(20, len(median_relevance))

    # median signed bar
    top_idx_m  = np.argsort(np.abs(median_relevance))[-n_show:]
    top_vals   = median_relevance[top_idx_m]
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.barh([f"N{i}" for i in top_idx_m], top_vals,
            color=["#C44E52" if v > 0 else "#4C72B0" for v in top_vals],
            edgecolor="white", linewidth=0.4)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlabel("Median Signed LRP Relevance", fontsize=11)
    ax.set_title(f"{layer_name} — Top {n_show} Neurons by Median Relevance\n"
                 "(red = promotes prediction | blue = suppresses)", fontsize=11, fontweight="bold")
    ax.invert_yaxis()
    ax.grid(True, axis='x', linestyle='--', alpha=0.4)
    plt.tight_layout()
    plt.savefig(os.path.join(cur_folder, "median_relevance_bar.png"), dpi=180)
    plt.close()

    # neuron relevance violin
    fig, ax = plt.subplots(figsize=(7, 5))
    parts = ax.violinplot(neuron_means, positions=[0], showmedians=True, showextrema=True)
    for pc in parts['bodies']:
        pc.set_facecolor("#4C72B0"); pc.set_alpha(0.7)
    parts['cmedians'].set_color("#C44E52"); parts['cmedians'].set_linewidth(2)
    ax.set_xticks([0])
    ax.set_xticklabels([layer_name.replace("_", " ")], fontsize=10)
    ax.set_ylabel("Mean Abs. Neuron Relevance", fontsize=10)
    ax.set_title(f"{layer_name} — Neuron Relevance Distribution\n"
                 f"({layer_array.shape[1]} neurons × {layer_array.shape[0]} patients)",
                 fontsize=11, fontweight="bold")
    ax.grid(True, linestyle='--', alpha=0.4)
    plt.tight_layout()
    plt.savefig(os.path.join(cur_folder, "neuron_relevance_violin.png"), dpi=180)
    plt.close()

    # top neuron heatmap per stage
    n_top_h     = min(20, layer_array.shape[1])
    top_h_idx   = np.argsort(np.abs(neuron_means))[-n_top_h:]
    stage_mat   = np.zeros((n_stages, n_top_h))
    for si, stage in enumerate(unique_stages):
        stage_mat[si] = np.abs(layer_array[labels == stage])[:, top_h_idx].mean(axis=0)

    fig, ax = plt.subplots(figsize=(14, 4))
    im = ax.imshow(stage_mat, aspect='auto', cmap='YlOrRd', interpolation='nearest')
    plt.colorbar(im, ax=ax, label="Mean Abs. Relevance")
    ax.set_yticks(range(n_stages))
    ax.set_yticklabels([f"Stage {s}" for s in unique_stages], fontsize=10)
    ax.set_xticks(range(n_top_h))
    ax.set_xticklabels([f"N{i}" for i in top_h_idx], rotation=45, ha='right', fontsize=8)
    ax.set_title(f"{layer_name} — Top {n_top_h} Neuron Relevance per Parkinson Stage",
                 fontsize=11, fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(cur_folder, "neuron_heatmap_per_stage.png"), dpi=180)
    plt.close()


# ══════════════════════════════════════════════════════════════
# RELEVANCE FLOW ACROSS LAYERS
# ══════════════════════════════════════════════════════════════
flow_folder = os.path.join(OUTPUT_FOLDER, "relevance_flow")
os.makedirs(flow_folder, exist_ok=True)

flow_layers = ["Input\nFeatures"] + \
              [ln.replace("_", "\n") for ln in layer_outputs] + \
              ["Prediction"]
flow_values = [np.mean(np.abs(all_relevance))] + \
              [np.mean(np.abs(np.array(v))) for v in layer_outputs.values()] + \
              [1.0]

fig, ax = plt.subplots(figsize=(10, 5))
x_pos = range(len(flow_layers))
ax.plot(x_pos, flow_values, marker='o', linewidth=2, color="#4C72B0", markersize=9, zorder=3)
ax.fill_between(x_pos, flow_values, alpha=0.15, color="#4C72B0")
for xp, yp in zip(x_pos, flow_values):
    ax.annotate(f"{yp:.4f}", (xp, yp), textcoords="offset points",
                xytext=(0, 12), ha='center', fontsize=9)
ax.set_xticks(x_pos)
ax.set_xticklabels(flow_layers, fontsize=10)
ax.set_ylabel("Mean Absolute Relevance", fontsize=11)
ax.set_xlabel("Network Stage", fontsize=11)
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig(os.path.join(flow_folder, "relevance_flow_across_layers.png"), dpi=180)
plt.close()

pd.DataFrame({"layer": flow_layers, "mean_absolute_relevance": flow_values}).to_csv(
    os.path.join(flow_folder, "relevance_flow_across_layers.csv"), index=False)


# ══════════════════════════════════════════════════════════════
# PER PATIENT RELEVANCE — CSV
# ══════════════════════════════════════════════════════════════
per_patient_df = pd.DataFrame(all_relevance, columns=feature_names)
per_patient_df = pd.concat([id_df.reset_index(drop=True), per_patient_df], axis=1)
per_patient_df[LABEL_COLUMN]      = labels
per_patient_df["predicted_class"] = pred_classes
per_patient_df["correct"]         = correct_mask
per_patient_df.to_csv(f"{OUTPUT_FOLDER}/relevance_per_patient.csv", index=False)


# ══════════════════════════════════════════════════════════════
# GLOBAL IMPORTANCE
# ══════════════════════════════════════════════════════════════
global_folder = os.path.join(OUTPUT_FOLDER, "global_importance")
os.makedirs(global_folder, exist_ok=True)
global_df.to_csv(f"{global_folder}/global_feature_importance.csv", index=False)

plot_top_features(global_df, TOP_N_GLOBAL, f"{global_folder}/top_global_features.png")

top_g    = global_df.head(TOP_N_GLOBAL).copy()
fig, ax  = plt.subplots(figsize=(9, 6))
ax.barh(top_g["feature"], top_g["mean_signed_relevance"],
        color=["#C44E52" if v > 0 else "#4C72B0" for v in top_g["mean_signed_relevance"]],
        edgecolor="white", linewidth=0.5)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Mean Signed LRP Relevance", fontsize=11)
ax.set_title(f"Top {TOP_N_GLOBAL} Global Features — Signed Relevance\n"
             "(red = promotes class | blue = suppresses class)", fontsize=12, fontweight="bold")
ax.invert_yaxis()
ax.grid(True, axis='x', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig(f"{global_folder}/top_global_features_signed.png", dpi=180)
plt.close()

sorted_abs = global_df.sort_values("mean_abs_relevance", ascending=True)
vmax_signed = sorted_abs["mean_signed_relevance"].abs().max()
fig, ax = plt.subplots(figsize=(8, max(6, len(feature_names) * 0.22)))
sc = ax.scatter(sorted_abs["mean_abs_relevance"], sorted_abs["feature"],
                c=sorted_abs["mean_signed_relevance"], cmap="RdBu_r", s=60,
                edgecolors="grey", linewidths=0.4,
                norm=plt.Normalize(vmin=-vmax_signed, vmax=vmax_signed))
plt.colorbar(plt.cm.ScalarMappable(cmap="RdBu_r",
             norm=plt.Normalize(vmin=-vmax_signed, vmax=vmax_signed)),
             ax=ax, label="Mean Signed Relevance")
ax.axvline(0, color="grey", linewidth=0.6, linestyle="--")
ax.set_xlabel("Mean Absolute LRP Relevance", fontsize=11)
ax.set_title("All Features — Global LRP Relevance\n"
             "(dot color = signed direction | position = magnitude)",
             fontsize=11, fontweight="bold")
ax.grid(True, axis='x', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig(f"{global_folder}/all_features_dot_plot.png", dpi=180)
plt.close()


# ══════════════════════════════════════════════════════════════
# PER CLASS IMPORTANCE
# ══════════════════════════════════════════════════════════════
class_importance = {}
per_class_folder = os.path.join(OUTPUT_FOLDER, "class_importance")
os.makedirs(per_class_folder, exist_ok=True)

for c in sorted(np.unique(labels)):
    mask     = labels == c
    rel      = all_relevance[mask]
    df_class = pd.DataFrame({
        "feature":               feature_names,
        "mean_signed_relevance": rel.mean(axis=0),
        "mean_abs_relevance":    np.abs(rel).mean(axis=0),
        "std_relevance":         np.abs(rel).std(axis=0),
    }).sort_values("mean_abs_relevance", ascending=False)
    df_class.to_csv(f"{per_class_folder}/class_{c}_feature_importance.csv", index=False)
    class_importance[c] = df_class

    plot_top_features(df_class, TOP_N_PER_CLASS,
                      f"{per_class_folder}/top_features_class_{c}.png")

    top_c   = df_class.head(TOP_N_PER_CLASS).copy()
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.barh(top_c["feature"], top_c["mean_signed_relevance"],
            xerr=top_c["std_relevance"],
            color=["#C44E52" if v > 0 else "#4C72B0" for v in top_c["mean_signed_relevance"]],
            edgecolor="white", linewidth=0.5,
            error_kw=dict(elinewidth=0.8, ecolor="gray", capsize=3))
    ax.axvline(0, color="black", linewidth=0.8)
    ax.invert_yaxis()
    ax.set_xlabel("Mean Signed LRP Relevance (± std)", fontsize=11)
    ax.set_title(f"Stage {c} — Top {TOP_N_PER_CLASS} Features  (n={mask.sum()} patients)",
                 fontsize=11, fontweight="bold")
    ax.grid(True, axis='x', linestyle='--', alpha=0.4)
    plt.tight_layout()
    plt.savefig(f"{per_class_folder}/class_{c}_signed_bar.png", dpi=180)
    plt.close()

fig, axes = plt.subplots(1, n_stages, figsize=(5 * n_stages, 7), sharey=False)
if n_stages == 1: axes = [axes]
for ax, stage in zip(axes, unique_stages):
    top_s = class_importance[stage].head(TOP_N_PER_CLASS)
    ax.barh(top_s["feature"], top_s["mean_signed_relevance"],
            color=["#C44E52" if v > 0 else "#4C72B0" for v in top_s["mean_signed_relevance"]],
            edgecolor='white', linewidth=0.4)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(f"Stage {stage}  (n={(labels==stage).sum()})",
                 fontsize=11, fontweight="bold", color=stage_colors[stage])
    ax.set_xlabel("Mean Signed Relevance", fontsize=9)
    ax.invert_yaxis()
    ax.tick_params(axis='y', labelsize=8)
plt.suptitle(f"Top {TOP_N_PER_CLASS} Feature Relevance Per Parkinson Stage (Signed)",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(per_class_folder, "all_stages_signed_small_multiples.png"), dpi=180)
plt.close()


# ══════════════════════════════════════════════════════════════
# STAGE PROGRESSION IMPORTANCE CURVE
# ══════════════════════════════════════════════════════════════
stage_feature_importance = np.zeros((n_stages, len(feature_names)))
for i, stage in enumerate(unique_stages):
    stage_feature_importance[i] = np.abs(all_relevance[labels == stage]).mean(axis=0)

top_features_indices         = np.argsort(global_mean_abs)[-TOP_N_FEATURES:]
top_feature_names            = [feature_names[i] for i in top_features_indices]
stage_feature_importance_top = stage_feature_importance[:, top_features_indices]

fig, ax = plt.subplots(figsize=(12, 6))
cmap_prog = plt.colormaps['tab10'].resampled(len(top_feature_names))
for i, feat_name in enumerate(top_feature_names):
    ax.plot(unique_stages, stage_feature_importance_top[:, i],
            marker='o', label=feat_name, color=cmap_prog(i), linewidth=2)
ax.set_xticks(unique_stages)
ax.set_xticklabels([f"Stage {s}" for s in unique_stages], fontsize=11)
ax.set_xlabel("Parkinson Stage (H&Y)", fontsize=11)
ax.set_ylabel("Mean Absolute LRP Relevance", fontsize=11)
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_FOLDER, "stage_progression_importance_curve.png"), dpi=180)
plt.close()


# ══════════════════════════════════════════════════════════════
# CORRECT vs MISCLASSIFIED
# ══════════════════════════════════════════════════════════════
correct_folder = os.path.join(OUTPUT_FOLDER, "correct_vs_misclassified")
os.makedirs(correct_folder, exist_ok=True)

correct_rel   = np.abs(all_relevance[correct_mask]).mean(axis=0)
incorrect_rel = np.abs(all_relevance[~correct_mask]).mean(axis=0)
diff_rel      = correct_rel - incorrect_rel
top_diff_idx  = np.argsort(np.abs(diff_rel))[-TOP_N_GLOBAL:]
top_diff_names = [feature_names[i] for i in top_diff_idx]

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
axes[0].barh(top_diff_names,  correct_rel[top_diff_idx],   color="#55A868", label="Correct")
axes[0].barh(top_diff_names, -incorrect_rel[top_diff_idx], color="#C44E52", label="Misclassified")
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].set_xlabel("Mean Absolute Relevance (←Misclassified | Correct→)")
axes[0].set_title("Feature Relevance: Correct vs Misclassified", fontweight="bold")
axes[0].legend(); axes[0].invert_yaxis()
axes[1].barh(top_diff_names, diff_rel[top_diff_idx],
             color=["#55A868" if d > 0 else "#C44E52" for d in diff_rel[top_diff_idx]])
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_xlabel("Δ Relevance (Correct − Misclassified)")
axes[1].set_title("Relevance Difference", fontweight="bold")
plt.suptitle(f"LRP Relevance: Correctly vs Incorrectly Classified Patients\n"
             f"(n_correct={correct_mask.sum()}, n_misclassified={(~correct_mask).sum()})",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(correct_folder, "correct_vs_misclassified_relevance.png"), dpi=180)
plt.close()

pd.DataFrame({
    "feature": feature_names, "correct_rel": correct_rel,
    "incorrect_rel": incorrect_rel, "delta": diff_rel,
}).sort_values("delta", ascending=False).to_csv(
    os.path.join(correct_folder, "correct_vs_misclassified_relevance.csv"), index=False)


# ══════════════════════════════════════════════════════════════
# VARIANCE vs RELEVANCE
# ══════════════════════════════════════════════════════════════
relevance_rank = np.argsort(np.argsort(global_mean_abs))
feat_variance  = X.var(axis=0)

fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(feat_variance, global_mean_abs, alpha=0.6, s=40,
                c=relevance_rank, cmap="viridis", edgecolors="none")
for idx in np.argsort(global_mean_abs)[-5:]:
    ax.annotate(feature_names[idx], (feat_variance[idx], global_mean_abs[idx]),
                fontsize=7, xytext=(4, 4), textcoords="offset points")
plt.colorbar(sc, ax=ax, label="Relevance rank")
ax.set_xlabel("Feature Variance (in dataset)", fontsize=11)
ax.set_ylabel("Mean Absolute LRP Relevance", fontsize=11)
ax.set_title("Feature Variance vs LRP Relevance\n"
             "(Do high-variance features get more credit?)", fontsize=12, fontweight="bold")
rho, p = spearmanr(feat_variance, global_mean_abs)
ax.text(0.05, 0.92, f"Spearman ρ = {rho:.3f}  (p={p:.3f})", transform=ax.transAxes,
        fontsize=10, bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray"))
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_FOLDER, "variance_vs_relevance.png"), dpi=180)
plt.close()


# ══════════════════════════════════════════════════════════════
# LAYER SPECIALIZATION ENTROPY
# ══════════════════════════════════════════════════════════════
layer_arrays = {name: np.array(vals) for name, vals in layer_outputs.items()}
layer_names  = list(layer_arrays.keys())

entropy_folder = os.path.join(OUTPUT_FOLDER, "layer_relevance", "plots")
os.makedirs(entropy_folder, exist_ok=True)

entropy_per_stage = {name: [] for name in layer_names}
for name, arr in layer_arrays.items():
    for stage in unique_stages:
        stage_mean = np.abs(arr[labels == stage]).mean(axis=0)
        prob       = stage_mean / (stage_mean.sum() + 1e-8)
        entropy_per_stage[name].append(scipy_entropy(prob + 1e-10))

x           = np.arange(n_stages)
width       = 0.8 / len(layer_names)
cmap_layers = plt.colormaps['Set2'].resampled(len(layer_names))

fig, ax = plt.subplots(figsize=(9, 5))
for i, name in enumerate(layer_names):
    offset = (i - len(layer_names) / 2 + 0.5) * width
    ax.bar(x + offset, entropy_per_stage[name], width=width, linewidth=0.5,
           label=name.replace("_", " "), color=cmap_layers(i), edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels([f"Stage {s}" for s in unique_stages], fontsize=11)
ax.set_xlabel("Parkinson Stage", fontsize=11)
ax.set_ylabel("Relevance Entropy (nats)", fontsize=11)
ax.set_title("Layer Specialization — Relevance Entropy per Layer per Stage\n"
             "(Low entropy = few neurons dominate | High entropy = distributed)",
             fontsize=12, fontweight="bold")
ax.legend(title="Layer", fontsize=9)
ax.grid(True, axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig(os.path.join(entropy_folder, "layer_specialization_entropy.png"), dpi=180)
plt.close()


# ══════════════════════════════════════════════════════════════
# CROSS-LAYER RELEVANCE PROPAGATION
# ══════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(10, 5))
cmap_l = plt.colormaps['tab10'].resampled(len(layer_names))
for i, (name, arr) in enumerate(layer_arrays.items()):
    ax.plot(unique_stages, [np.abs(arr[labels == s]).mean() for s in unique_stages],
            marker='o', linewidth=2, label=name.replace("_", " "), color=cmap_l(i))
ax.set_xticks(unique_stages)
ax.set_xticklabels([f"Stage {s}" for s in unique_stages], fontsize=11)
ax.set_xlabel("Parkinson Stage", fontsize=11)
ax.set_ylabel("Mean Absolute Relevance", fontsize=11)
ax.set_title("Cross-Layer Relevance Propagation Across Disease Stages\n"
             "(How much relevance each layer carries per stage)",
             fontsize=12, fontweight="bold")
ax.legend(title="Layer", fontsize=9)
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig(os.path.join(entropy_folder, "cross_layer_relevance_propagation.png"), dpi=180)
plt.close()


# ══════════════════════════════════════════════════════════════
# EXTRA PLOTS
# ══════════════════════════════════════════════════════════════
extra_plots = os.path.join(OUTPUT_FOLDER, "extra_plots")
os.makedirs(extra_plots, exist_ok=True)

extra_top_idx   = np.argsort(global_mean_abs)[-TOP_N_FEATURES:]
extra_top_names = [feature_names[i] for i in extra_top_idx]


# feature fingerprint radar
stage_profiles = {}
for stage in unique_stages:
    profile = np.abs(all_relevance[labels == stage])[:, extra_top_idx].mean(axis=0)
    mx = profile.max()
    stage_profiles[stage] = profile / mx if mx > 0 else profile

angles = np.linspace(0, 2 * np.pi, TOP_N_FEATURES, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
for stage in unique_stages:
    vals = stage_profiles[stage].tolist() + stage_profiles[stage][:1].tolist()
    ax.plot(angles, vals, linewidth=2.5, label=f"Stage {stage}", color=stage_colors[stage])
    ax.fill(angles, vals, alpha=0.08, color=stage_colors[stage])
ax.set_thetagrids(np.degrees(angles[:-1]), extra_top_names, fontsize=9)
ax.set_ylim(0, 1)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(["25%", "50%", "75%", "100%"], fontsize=7, color="grey")
ax.set_title("Feature Relevance Fingerprint per Parkinson Stage\n"
             "(normalized — shape reveals each stage's biomarker signature)",
             fontsize=12, fontweight="bold", pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(extra_plots, "feature_fingerprint_radar.png"),
            dpi=180, bbox_inches='tight')
plt.close()


# stage reasoning dissimilarity matrix
stage_mean_vecs = {
    stage: np.abs(all_relevance[labels == stage]).mean(axis=0)
    for stage in unique_stages
}
dissimilarity = np.zeros((n_stages, n_stages))
for i, si in enumerate(unique_stages):
    for j, sj in enumerate(unique_stages):
        if i != j:
            dissimilarity[i, j] = cosine(stage_mean_vecs[si], stage_mean_vecs[sj])

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(dissimilarity, cmap='YlOrRd_r', vmin=0, vmax=dissimilarity.max())
plt.colorbar(im, ax=ax, label="Cosine Distance  (0 = identical reasoning)")
stage_labels = [f"Stage {s}" for s in unique_stages]
ax.set_xticks(range(n_stages)); ax.set_xticklabels(stage_labels, rotation=45, ha='right', fontsize=10)
ax.set_yticks(range(n_stages)); ax.set_yticklabels(stage_labels, fontsize=10)
for i in range(n_stages):
    for j in range(n_stages):
        ax.text(j, i, f"{dissimilarity[i,j]:.3f}", ha='center', va='center',
                fontsize=10, color="white" if dissimilarity[i, j] < 0.15 else "black")
ax.set_title("Stage Reasoning Dissimilarity Matrix\n"
             "(how differently does the model explain each pair of stages?)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(extra_plots, "stage_dissimilarity_matrix.png"), dpi=180)
plt.close()


# relevance direction per stage
fig, axes = plt.subplots(n_stages, 1, figsize=(11, 2.8 * n_stages), sharex=True)
if n_stages == 1: axes = [axes]

for ax, stage in zip(axes, unique_stages):
    mask    = labels == stage
    rel_sub = all_relevance[mask][:, extra_top_idx]
    pct_pos = (rel_sub > 0).mean(axis=0) * 100
    pct_neg = (rel_sub < 0).mean(axis=0) * 100
    x = np.arange(TOP_N_FEATURES)
    ax.bar(x,  pct_pos, color="#55A868", width=0.6)
    ax.bar(x, -pct_neg, color="#C44E52", width=0.6)
    ax.axhline(0,   color='black', linewidth=0.8)
    ax.axhline( 50, color='grey',  linewidth=0.5, linestyle='--')
    ax.axhline(-50, color='grey',  linewidth=0.5, linestyle='--')
    ax.set_ylim(-105, 105)
    ax.set_yticks([-100, -50, 0, 50, 100])
    ax.set_yticklabels(['100%↓', '50%↓', '0', '50%↑', '100%↑'], fontsize=8)
    ax.set_ylabel(f"Stage {stage}  (n={mask.sum()})", fontsize=10,
                  fontweight="bold", color=stage_colors[stage])
    ax.grid(True, axis='y', linestyle='--', alpha=0.3)

axes[-1].set_xticks(np.arange(TOP_N_FEATURES))
axes[-1].set_xticklabels(extra_top_names, rotation=30, ha='right', fontsize=9)
fig.legend(handles=[
    mpatches.Patch(color="#55A868", label="Positive — promotes stage prediction"),
    mpatches.Patch(color="#C44E52", label="Negative — suppresses stage prediction"),
], loc='upper right', fontsize=9, bbox_to_anchor=(1.0, 1.0))
plt.suptitle("Feature Relevance Direction per Stage\n"
             "(↑ = % patients where feature promotes prediction  |  "
             "↓ = % patients where feature suppresses prediction)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(extra_plots, "relevance_direction_per_stage.png"),
            dpi=180, bbox_inches='tight')
plt.close()


# layer-wise feature agreement
TOP_FEATURES_LAYER = 6
fig, axes = plt.subplots(1, len(layer_names), figsize=(5 * len(layer_names), 6), sharey=False)
if len(layer_names) == 1: axes = [axes]

for ax, (layer_name, layer_arr) in zip(axes, layer_arrays.items()):
    top_neurons       = np.argsort(np.abs(layer_arr).mean(axis=0))[-10:]
    top_neuron_signal = np.abs(layer_arr[:, top_neurons]).sum(axis=1)
    feat_corrs        = np.array([
        np.corrcoef(all_relevance[:, fi], top_neuron_signal)[0, 1]
        for fi in range(len(feature_names))
    ])
    feat_corrs = np.nan_to_num(feat_corrs)

    top_f_idx   = np.argsort(np.abs(feat_corrs))[-TOP_FEATURES_LAYER:]
    top_f_names = [feature_names[i] for i in top_f_idx]
    top_f_corrs = feat_corrs[top_f_idx]

    ax.barh(top_f_names, top_f_corrs,
            color=["#C44E52" if v > 0 else "#4C72B0" for v in top_f_corrs],
            edgecolor='white', linewidth=0.5)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlabel("Correlation with layer\ntop neuron activity", fontsize=9)
    ax.set_title(f"{layer_name.replace('_', ' ')}\n({layer_arr.shape[1]} neurons)",
                 fontsize=10, fontweight="bold")
    ax.invert_yaxis()
    ax.set_xlim(-1, 1)
    ax.grid(True, axis='x', linestyle='--', alpha=0.4)
    ax.tick_params(axis='y', labelsize=8)

fig.legend(handles=[
    mpatches.Patch(color="#C44E52", label="Positive — feature activates layer"),
    mpatches.Patch(color="#4C72B0", label="Negative — feature suppresses layer"),
], loc='lower center', ncol=2, fontsize=9, bbox_to_anchor=(0.5, -0.04))
plt.suptitle("Layer-wise Feature Agreement\n"
             "(which input features drive each layer's most active neurons?)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(extra_plots, "layerwise_feature_agreement.png"),
            dpi=180, bbox_inches='tight')
plt.close()


print("\nLRP analysis complete. Results saved in:", OUTPUT_FOLDER)

KeyboardInterrupt: 

##### with BatchNorm1d layer (old)

In [5]:
# create a copy of mlp without BatchNorm for LRP
class LRPMLPWrapper(nn.Module):
    def __init__(self, mlp, classifier):
        super().__init__()
        # filter out BatchNorm layers
        self.layers = nn.Sequential(*(l for l in mlp if not isinstance(l, nn.BatchNorm1d)))
        self.classifier = classifier

    def forward(self, x):
        features = self.layers(x)
        return self.classifier(features)

# wrapper
lrp_model = LRPMLPWrapper(model.mlp, model.classifier)

# LRP layer setup
layers_to_explain = [
    ("layer1_linear", lrp_model.layers[0]),
    ("layer2_linear", lrp_model.layers[3]),
    ("classifier", lrp_model.classifier)
]

layer_lrps = {name: LayerLRP(lrp_model, layer) for name, layer in layers_to_explain}
layer_outputs = {name: [] for name, _ in layers_to_explain}


# LRP computation
lrp = LRP(lrp_model)  # explain model with respect to inputs

all_relevance = []
pred_classes = []

for i in range(X_tensor.shape[0]):
    x = X_tensor[i].unsqueeze(0)

    # predict class (use full model or wrapper)
    pred_class = lrp_model(x).argmax(dim=1).item()
    pred_classes.append(pred_class)

    # full LRP for input features
    relevance = lrp.attribute(x, target=pred_class)
    relevance = relevance.detach().cpu().numpy().flatten()
    all_relevance.append(relevance)

    # per-layer relevance
    for layer_name, layer_lrp in layer_lrps.items():
        # call attribute on same wrapper and same input
        layer_rel = layer_lrp.attribute(x, target=pred_class)
        layer_rel = layer_rel.detach().cpu().numpy().flatten()
        layer_outputs[layer_name].append(layer_rel)

all_relevance = np.array(all_relevance)
pred_classes = np.array(pred_classes)
correct_mask  = pred_classes == labels


# 1. save LAYER RELEVANCE (patient + median)
layer_folder = os.path.join(OUTPUT_FOLDER, "layer_relevance")
os.makedirs(layer_folder, exist_ok=True)

for layer_name, values in layer_outputs.items():
    layer_array = np.array(values)

    current_layer_folder = os.path.join(layer_folder, layer_name)
    os.makedirs(current_layer_folder, exist_ok=True)
    # dataframe per patient
    layer_df = pd.DataFrame(layer_array)
    layer_df = pd.concat([id_df.reset_index(drop=True), layer_df], axis=1)
    layer_df.to_csv(os.path.join(current_layer_folder, "relevance_per_patient.csv"), index=False)

    # median relevance (global behaviour)
    median_relevance = np.median(layer_array, axis=0)
    median_df = pd.DataFrame({
        "neuron_index": range(len(median_relevance)),
        "median_relevance": median_relevance
    })
    median_df.to_csv(os.path.join(current_layer_folder, "median_relevance.csv"), index=False)


    # 2. save RELEVANCE FLOW across layers
    flow_folder = os.path.join(OUTPUT_FOLDER, "relevance_flow")
    os.makedirs(flow_folder, exist_ok=True)

    flow_layers = ["Input\nFeatures"]
    flow_values = [np.mean(np.abs(all_relevance))]

    for layer_name, values in layer_outputs.items():
        layer_array = np.array(values)

        clean_name = layer_name.replace("_", "\n")

        flow_layers.append(clean_name)
        flow_values.append(np.mean(np.abs(layer_array)))

    flow_layers.append("Prediction")
    flow_values.append(1.0)

    fig, ax = plt.subplots(figsize=(10, 5))
    x_pos = range(len(flow_layers))
    ax.plot(x_pos, flow_values, marker='o', linewidth=2, color="#4C72B0", markersize=9, zorder=3)
    ax.fill_between(x_pos, flow_values, alpha=0.15, color="#4C72B0")
    for x, y in zip(x_pos, flow_values):
        ax.annotate(f"{y:.4f}", (x, y), textcoords="offset points", xytext=(0, 12), ha='center', fontsize=9)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(flow_layers, fontsize=10)
    ax.set_ylabel("Mean Absolute Relevance", fontsize=11)
    ax.set_xlabel("Network Stage", fontsize=11)
    ax.grid(True, linestyle='--', alpha=0.4)
    plt.tight_layout()
    plt.savefig(os.path.join(flow_folder, "relevance_flow_across_layers.png"), dpi=180)
    plt.close()

    pd.DataFrame({
        "layer": flow_layers,
        "mean_absolute_relevance": flow_values
    }).to_csv(os.path.join(flow_folder, "relevance_flow_across_layers.csv"), index=False)


# 3. save PER PATIENT RELEVANCE
per_patient_df = pd.DataFrame(all_relevance, columns=feature_names)
per_patient_df = pd.concat([id_df.reset_index(drop=True), per_patient_df], axis=1)
per_patient_df[LABEL_COLUMN] = labels
per_patient_df["predicted_class"] = pred_classes
per_patient_df["correct"] = correct_mask

per_patient_df.to_csv(f"{OUTPUT_FOLDER}/relevance_per_patient.csv", index=False)


# 4. save GLOBAL IMPORTANCE
global_mean_signed = all_relevance.mean(axis=0)
global_mean_abs = np.abs(all_relevance).mean(axis=0)

global_df = pd.DataFrame({
    "feature": feature_names,
    "mean_signed_relevance": global_mean_signed,
    "mean_abs_relevance": global_mean_abs
}).sort_values("mean_abs_relevance", ascending=False)


global_folder = os.path.join(OUTPUT_FOLDER, "global_importance")
os.makedirs(global_folder, exist_ok=True)
global_df.to_csv(f"{global_folder}/global_feature_importance.csv", index=False)

# global plot
plot_top_features(
    global_df,
    TOP_N_GLOBAL,
    f"{global_folder}/top_global_features.png"
)


#5. save PER CLASS IMPORTANCE
class_importance = {}

per_class_folder = os.path.join(OUTPUT_FOLDER, "class_importance")
os.makedirs(per_class_folder, exist_ok=True)

for c in sorted(np.unique(labels)):
    class_mask = labels == c
    class_rel = all_relevance[class_mask]

    mean_signed = class_rel.mean(axis=0)
    mean_abs = np.abs(class_rel).mean(axis=0)

    df_class = pd.DataFrame({
        "feature": feature_names,
        "mean_signed_relevance": mean_signed,
        "mean_abs_relevance": mean_abs
    }).sort_values("mean_abs_relevance", ascending=False)

    df_class.to_csv(f"{per_class_folder}/class_{c}_feature_importance.csv", index=False)
    class_importance[c] = df_class

# per class plots
for c, df_class in class_importance.items():
    plot_top_features(
        df_class,
        TOP_N_PER_CLASS,
        f"{per_class_folder}/top_features_class_{c}.png"
    )

# 6. save STAGE PROGRESSION IMPORTANCE curve
unique_stages = sorted(np.unique(labels))

stage_feature_importance = np.zeros((len(unique_stages), len(feature_names)))

for i, stage in enumerate(unique_stages):
    mask = labels == stage
    stage_feature_importance[i] = np.abs(all_relevance[mask]).mean(axis=0)

top_features_indices  = np.argsort(global_mean_abs)[-TOP_N_FEATURES:]
top_feature_names     = [feature_names[i] for i in top_features_indices]
stage_feature_importance_top = stage_feature_importance[:, top_features_indices]

fig, ax = plt.subplots(figsize=(12, 6))
cmap_prog = plt.colormaps['tab10'].resampled(len(top_feature_names))

for i, feature_name in enumerate(top_feature_names):
    ax.plot(unique_stages, stage_feature_importance_top[:, i],
            marker='o', label=feature_name, color=cmap_prog(i), linewidth=2)

ax.set_xticks(unique_stages)
ax.set_xticklabels([f"Stage {s}" for s in unique_stages], fontsize=11)
ax.set_xlabel("Parkinson Stage", fontsize=11)
ax.set_ylabel("Mean Absolute LRP Relevance", fontsize=11)
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_FOLDER, "stage_progression_importance_curve.png"), dpi=180)
plt.close()


# 7. save CORRECT vs MISCLASSIFIED RELEVANCE comparison
correct_folder = os.path.join(OUTPUT_FOLDER, "correct_vs_misclassified")
os.makedirs(correct_folder, exist_ok=True)

correct_rel = np.abs(all_relevance[correct_mask]).mean(axis=0)
incorrect_rel = np.abs(all_relevance[~correct_mask]).mean(axis=0)
diff_rel = correct_rel - incorrect_rel

top_diff_idx = np.argsort(np.abs(diff_rel))[-TOP_N_GLOBAL:]
top_diff_names = [feature_names[i] for i in top_diff_idx]

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
bar_colors = ["#55A868" if d > 0 else "#C44E52" for d in diff_rel[top_diff_idx]]

axes[0].barh(top_diff_names, correct_rel[top_diff_idx],    color="#55A868", label="Correct")
axes[0].barh(top_diff_names, -incorrect_rel[top_diff_idx], color="#C44E52", label="Misclassified")
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].set_xlabel("Mean Absolute Relevance (←Misclassified | Correct→)")
axes[0].set_title("Feature Relevance: Correct vs Misclassified", fontweight="bold")
axes[0].legend()
axes[0].invert_yaxis()

axes[1].barh(top_diff_names, diff_rel[top_diff_idx], color=bar_colors)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_xlabel("Δ Relevance (Correct - Misclassified)")
axes[1].set_title("Relevance Difference", fontweight="bold")

plt.suptitle(
    f"LRP Relevance: Correctly vs Incorrectly Classified Patients\n"
    f"(n_correct={correct_mask.sum()}, n_misclassified={(~correct_mask).sum()})",
    fontsize=12, fontweight="bold"
)
plt.tight_layout()
plt.savefig(os.path.join(correct_folder, "correct_vs_misclassified_relevance.png"), dpi=180)
plt.close()

pd.DataFrame({
    "feature":       feature_names,
    "correct_rel":   correct_rel,
    "incorrect_rel": incorrect_rel,
    "delta":         diff_rel,
}).sort_values("delta", ascending=False).to_csv(
    os.path.join(correct_folder, "correct_vs_misclassified_relevance.csv"), index=False
)


# 8. save VARIANCE vs RELEVANCE
relevance_rank = np.argsort(np.argsort(global_mean_abs))  # rank 0 = least important
feat_variance  = X.var(axis=0)

fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(feat_variance, global_mean_abs, alpha=0.6, s=20, c=relevance_rank, cmap="viridis")
plt.colorbar(sc, ax=ax, label="Relevance rank")
ax.set_xlabel("Feature Variance (in dataset)", fontsize=11)
ax.set_ylabel("Mean Absolute LRP Relevance", fontsize=11)
ax.set_title(
    "Feature Variance vs LRP Relevance\n(Do high-variance features get more credit?)",
    fontsize=12, fontweight="bold"
)
rho, p = spearmanr(feat_variance, global_mean_abs)
ax.text(
    0.05, 0.92, f"Spearman ρ = {rho:.3f}  (p={p:.3f})",
    transform=ax.transAxes, fontsize=10, color="black",
    bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray")
)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_FOLDER, "variance_vs_relevance.png"), dpi=180)
plt.close()


# 9. save RELEVANCE ENTROPY PER LAYER
layer_arrays = {name: np.array(vals) for name, vals in layer_outputs.items()}
layer_names  = list(layer_arrays.keys())

entropy_folder = os.path.join(OUTPUT_FOLDER, "layer_relevance", "plots")
os.makedirs(entropy_folder, exist_ok=True)

fig, ax = plt.subplots(figsize=(9, 5))

entropy_per_stage = {name: [] for name in layer_names}

for name, arr in layer_arrays.items():
    abs_arr = np.abs(arr)
    for stage in unique_stages:
        mask = labels == stage
        stage_mean = abs_arr[mask].mean(axis=0)
        prob = stage_mean / (stage_mean.sum() + 1e-8)
        ent = scipy_entropy(prob + 1e-10)
        entropy_per_stage[name].append(ent)

x = np.arange(len(unique_stages))
width = 0.8 / len(layer_names)
cmap_layers = plt.colormaps['Set2'].resampled(len(layer_names))

for i, name in enumerate(layer_names):
    offset = (i - len(layer_names) / 2 + 0.5) * width
    ax.bar(
        x + offset, entropy_per_stage[name], width=width, linewidth=0.5,
        label=name.replace("_", " "), color=cmap_layers(i), edgecolor='white' 
    )

ax.set_xticks(x)
ax.set_xticklabels([f"Stage {s}" for s in unique_stages], fontsize=11)
ax.set_xlabel("Parkinson Stage", fontsize=11)
ax.set_ylabel("Relevance Entropy (nats)", fontsize=11)
ax.set_title(
    "Layer Specialization — Relevance Entropy per Layer per Stage\n"
    "(Low entropy = few neurons dominate | High entropy = distributed)",
    fontsize=12, fontweight="bold"
)
ax.legend(title="Layer", fontsize=9)
ax.grid(True, axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig(os.path.join(entropy_folder, "layer_specialization_entropy.png"), dpi=180)
plt.close()


# 10. save CROSS-LAYER RELEVANCE PROPAGATION
fig, ax = plt.subplots(figsize=(10, 5))
cmap_l = plt.colormaps['tab10'].resampled(len(layer_names))

for i, (name, arr) in enumerate(layer_arrays.items()):
    abs_arr     = np.abs(arr)
    stage_means = [abs_arr[labels == s].mean() for s in unique_stages]
    ax.plot(
        unique_stages, stage_means, marker='o', linewidth=2,
        label=name.replace("_", " "), color=cmap_l(i)
    )

ax.set_xticks(unique_stages)
ax.set_xticklabels([f"Stage {s}" for s in unique_stages], fontsize=11)
ax.set_xlabel("Parkinson Stage", fontsize=11)
ax.set_ylabel("Mean Absolute Relevance", fontsize=11)
ax.set_title(
    "Cross-Layer Relevance Propagation Across Disease Stages\n"
    "(How much relevance each layer carries per stage)",
    fontsize=12, fontweight="bold"
)
ax.legend(title="Layer", fontsize=9)
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig(os.path.join(entropy_folder, "cross_layer_relevance_propagation.png"), dpi=180)
plt.close()


print("\nLRP analysis complete, results saved in:", OUTPUT_FOLDER)

d:\USTHB\M2\Stage Parkinson\Parkinson-prediction\.venv\Lib\site-packages\captum\attr\_core\lrp.py:207: UserWarning: Input Tensor 0 did not already require gradients, required_grads has been set automatically.
  gradient_mask = apply_gradient_requirements(input_tuple)
d:\USTHB\M2\Stage Parkinson\Parkinson-prediction\.venv\Lib\site-packages\captum\attr\_core\layer\layer_lrp.py:233: UserWarning: Input Tensor 0 did not already require gradients, required_grads has been set automatically.
  gradient_mask = apply_gradient_requirements(inputs_tuple)



LRP analysis complete, results saved in: lrp_analysis_30_v4


##### without BatchNorm1d layer (old)

In [ ]:
# LRP layer setup
layers_to_explain = [
    ("layer1_linear", model.mlp[0]),
    ("layer2_linear", model.mlp[3]),
    ("classifier", model.classifier)
]
layer_lrps = {name: LayerLRP(model, layer) for name, layer in layers_to_explain}
layer_outputs = {name: [] for name, _ in layers_to_explain}


# LRP computation
lrp = LRP(model)

all_relevance = []
pred_classes  = []

for i in range(X_tensor.shape[0]):
    x = X_tensor[i].unsqueeze(0)   # adds batch dimension; (num_features,) -> (1, num_features)
    pred_class = model(x).argmax(dim=1).item()
    pred_classes.append(pred_class)

    relevance = lrp.attribute(x, target=pred_class)
    relevance = relevance.detach().cpu().numpy().flatten()   # convert tensor to numpy array; (1, num_features) -> (num_features,)

    # LRP relevance per layer
    for layer_name, layer_lrp in layer_lrps.items():
        layer_rel = layer_lrp.attribute(x, target=pred_class)
        layer_rel = layer_rel.detach().cpu().numpy().flatten()
        layer_outputs[layer_name].append(layer_rel)

    all_relevance.append(relevance)

all_relevance = np.array(all_relevance)
pred_classes  = np.array(pred_classes)
correct_mask  = pred_classes == labels


# 1. save LAYER RELEVANCE (patient + median)
layer_folder = os.path.join(OUTPUT_FOLDER, "layer_relevance")
os.makedirs(layer_folder, exist_ok=True)

for layer_name, values in layer_outputs.items():
    layer_array = np.array(values)

    current_layer_folder = os.path.join(layer_folder, layer_name)
    os.makedirs(current_layer_folder, exist_ok=True)
    # dataframe per patient
    layer_df = pd.DataFrame(layer_array)
    layer_df = pd.concat([id_df.reset_index(drop=True), layer_df], axis=1)
    layer_df.to_csv(os.path.join(current_layer_folder, "relevance_per_patient.csv"), index=False)

    # median relevance (global behaviour)
    median_relevance = np.median(layer_array, axis=0)
    median_df = pd.DataFrame({
        "neuron_index": range(len(median_relevance)),
        "median_relevance": median_relevance
    })
    median_df.to_csv(os.path.join(current_layer_folder, "median_relevance.csv"), index=False)


    # 2. save RELEVANCE FLOW across layers
    flow_folder = os.path.join(OUTPUT_FOLDER, "relevance_flow")
    os.makedirs(flow_folder, exist_ok=True)

    flow_layers = ["Input\nFeatures"]
    flow_values = [np.mean(np.abs(all_relevance))]

    for layer_name, values in layer_outputs.items():
        layer_array = np.array(values)

        clean_name = layer_name.replace("_", "\n")

        flow_layers.append(clean_name)
        flow_values.append(np.mean(np.abs(layer_array)))

    flow_layers.append("Prediction")
    flow_values.append(1.0)

    fig, ax = plt.subplots(figsize=(10, 5))
    x_pos = range(len(flow_layers))
    ax.plot(x_pos, flow_values, marker='o', linewidth=2, color="#4C72B0", markersize=9, zorder=3)
    ax.fill_between(x_pos, flow_values, alpha=0.15, color="#4C72B0")
    for x, y in zip(x_pos, flow_values):
        ax.annotate(f"{y:.4f}", (x, y), textcoords="offset points", xytext=(0, 12), ha='center', fontsize=9)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(flow_layers, fontsize=10)
    ax.set_ylabel("Mean Absolute Relevance", fontsize=11)
    ax.set_xlabel("Network Stage", fontsize=11)
    ax.grid(True, linestyle='--', alpha=0.4)
    plt.tight_layout()
    plt.savefig(os.path.join(flow_folder, "relevance_flow_across_layers.png"), dpi=180)
    plt.close()

    pd.DataFrame({
        "layer": flow_layers,
        "mean_absolute_relevance": flow_values
    }).to_csv(os.path.join(flow_folder, "relevance_flow_across_layers.csv"), index=False)


# 3. save PER PATIENT RELEVANCE
per_patient_df = pd.DataFrame(all_relevance, columns=feature_names)
per_patient_df = pd.concat([id_df.reset_index(drop=True), per_patient_df], axis=1)
per_patient_df[LABEL_COLUMN] = labels
per_patient_df["predicted_class"] = pred_classes
per_patient_df["correct"] = correct_mask

per_patient_df.to_csv(f"{OUTPUT_FOLDER}/relevance_per_patient.csv", index=False)


# 4. save GLOBAL IMPORTANCE
global_mean_signed = all_relevance.mean(axis=0)
global_mean_abs = np.abs(all_relevance).mean(axis=0)

global_df = pd.DataFrame({
    "feature": feature_names,
    "mean_signed_relevance": global_mean_signed,
    "mean_abs_relevance": global_mean_abs
}).sort_values("mean_abs_relevance", ascending=False)


global_folder = os.path.join(OUTPUT_FOLDER, "global_importance")
os.makedirs(global_folder, exist_ok=True)
global_df.to_csv(f"{global_folder}/global_feature_importance.csv", index=False)

# global plot
plot_top_features(
    global_df,
    TOP_N_GLOBAL,
    f"{global_folder}/top_global_features.png"
)


#5. save PER CLASS IMPORTANCE
class_importance = {}

per_class_folder = os.path.join(OUTPUT_FOLDER, "class_importance")
os.makedirs(per_class_folder, exist_ok=True)

for c in sorted(np.unique(labels)):
    class_mask = labels == c
    class_rel = all_relevance[class_mask]

    mean_signed = class_rel.mean(axis=0)
    mean_abs = np.abs(class_rel).mean(axis=0)

    df_class = pd.DataFrame({
        "feature": feature_names,
        "mean_signed_relevance": mean_signed,
        "mean_abs_relevance": mean_abs
    }).sort_values("mean_abs_relevance", ascending=False)

    df_class.to_csv(f"{per_class_folder}/class_{c}_feature_importance.csv", index=False)
    class_importance[c] = df_class

# per class plots
for c, df_class in class_importance.items():
    plot_top_features(
        df_class,
        TOP_N_PER_CLASS,
        f"{per_class_folder}/top_features_class_{c}.png"
    )

# 6. save STAGE PROGRESSION IMPORTANCE curve
unique_stages = sorted(np.unique(labels))

stage_feature_importance = np.zeros((len(unique_stages), len(feature_names)))

for i, stage in enumerate(unique_stages):
    mask = labels == stage
    stage_feature_importance[i] = np.abs(all_relevance[mask]).mean(axis=0)

top_features_indices  = np.argsort(global_mean_abs)[-TOP_N_FEATURES:]
top_feature_names     = [feature_names[i] for i in top_features_indices]
stage_feature_importance_top = stage_feature_importance[:, top_features_indices]

fig, ax = plt.subplots(figsize=(12, 6))
cmap_prog = plt.colormaps['tab10'].resampled(len(top_feature_names))

for i, feature_name in enumerate(top_feature_names):
    ax.plot(unique_stages, stage_feature_importance_top[:, i],
            marker='o', label=feature_name, color=cmap_prog(i), linewidth=2)

ax.set_xticks(unique_stages)
ax.set_xticklabels([f"Stage {s}" for s in unique_stages], fontsize=11)
ax.set_xlabel("Parkinson Stage", fontsize=11)
ax.set_ylabel("Mean Absolute LRP Relevance", fontsize=11)
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_FOLDER, "stage_progression_importance_curve.png"), dpi=180)
plt.close()


# 7. save CORRECT vs MISCLASSIFIED RELEVANCE comparison
correct_folder = os.path.join(OUTPUT_FOLDER, "correct_vs_misclassified")
os.makedirs(correct_folder, exist_ok=True)

correct_rel = np.abs(all_relevance[correct_mask]).mean(axis=0)
incorrect_rel = np.abs(all_relevance[~correct_mask]).mean(axis=0)

diff_rel = correct_rel - incorrect_rel
top_diff_idx = np.argsort(np.abs(diff_rel))[-TOP_N_GLOBAL:]
top_diff_names = [feature_names[i] for i in top_diff_idx]

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
bar_colors = ["#55A868" if d > 0 else "#C44E52" for d in diff_rel[top_diff_idx]]

axes[0].barh(top_diff_names, correct_rel[top_diff_idx],    color="#55A868", label="Correct")
axes[0].barh(top_diff_names, -incorrect_rel[top_diff_idx], color="#C44E52", label="Misclassified")
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].set_xlabel("Mean Absolute Relevance (←Misclassified | Correct→)")
axes[0].set_title("Feature Relevance: Correct vs Misclassified", fontweight="bold")
axes[0].legend()
axes[0].invert_yaxis()

axes[1].barh(top_diff_names, diff_rel[top_diff_idx], color=bar_colors)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_xlabel("Δ Relevance (Correct - Misclassified)")
axes[1].set_title("Relevance Difference", fontweight="bold")

plt.suptitle(
    f"LRP Relevance: Correctly vs Incorrectly Classified Patients\n"
    f"(n_correct={correct_mask.sum()}, n_misclassified={(~correct_mask).sum()})",
    fontsize=12, fontweight="bold"
)
plt.tight_layout()
plt.savefig(os.path.join(correct_folder, "correct_vs_misclassified_relevance.png"), dpi=180)
plt.close()

pd.DataFrame({
    "feature":       feature_names,
    "correct_rel":   correct_rel,
    "incorrect_rel": incorrect_rel,
    "delta":         diff_rel,
}).sort_values("delta", ascending=False).to_csv(
    os.path.join(correct_folder, "correct_vs_misclassified_relevance.csv"), index=False
)


# 8. save VARIANCE vs RELEVANCE
relevance_rank = np.argsort(np.argsort(global_mean_abs))  # rank 0 = least important
feat_variance  = X.var(axis=0)

fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(feat_variance, global_mean_abs, alpha=0.6, s=20, c=relevance_rank, cmap="viridis")
plt.colorbar(sc, ax=ax, label="Relevance rank")
ax.set_xlabel("Feature Variance (in dataset)", fontsize=11)
ax.set_ylabel("Mean Absolute LRP Relevance", fontsize=11)
ax.set_title(
    "Feature Variance vs LRP Relevance\n(Do high-variance features get more credit?)",
    fontsize=12, fontweight="bold"
)
rho, p = spearmanr(feat_variance, global_mean_abs)
ax.text(
    0.05, 0.92, f"Spearman ρ = {rho:.3f}  (p={p:.3f})",
    transform=ax.transAxes, fontsize=10, color="black",
    bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray")
)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_FOLDER, "variance_vs_relevance.png"), dpi=180)
plt.close()


# 9. save RELEVANCE ENTROPY PER LAYER
layer_arrays = {name: np.array(vals) for name, vals in layer_outputs.items()}
layer_names  = list(layer_arrays.keys())

entropy_folder = os.path.join(OUTPUT_FOLDER, "layer_relevance", "plots")
os.makedirs(entropy_folder, exist_ok=True)

fig, ax = plt.subplots(figsize=(9, 5))

entropy_per_stage = {name: [] for name in layer_names}

for name, arr in layer_arrays.items():
    abs_arr = np.abs(arr)
    for stage in unique_stages:
        mask = labels == stage
        stage_mean = abs_arr[mask].mean(axis=0)
        prob = stage_mean / (stage_mean.sum() + 1e-8)
        ent = scipy_entropy(prob + 1e-10)
        entropy_per_stage[name].append(ent)

x = np.arange(len(unique_stages))
width = 0.8 / len(layer_names)
cmap_layers = plt.colormaps['Set2'].resampled(len(layer_names))

for i, name in enumerate(layer_names):
    offset = (i - len(layer_names) / 2 + 0.5) * width
    ax.bar(
        x + offset, entropy_per_stage[name], width=width, linewidth=0.5,
        label=name.replace("_", " "), color=cmap_layers(i), edgecolor='white' 
    )

ax.set_xticks(x)
ax.set_xticklabels([f"Stage {s}" for s in unique_stages], fontsize=11)
ax.set_xlabel("Parkinson Stage", fontsize=11)
ax.set_ylabel("Relevance Entropy (nats)", fontsize=11)
ax.set_title(
    "Layer Specialization — Relevance Entropy per Layer per Stage\n"
    "(Low entropy = few neurons dominate | High entropy = distributed)",
    fontsize=12, fontweight="bold"
)
ax.legend(title="Layer", fontsize=9)
ax.grid(True, axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig(os.path.join(entropy_folder, "layer_specialization_entropy.png"), dpi=180)
plt.close()


# 10. save CROSS-LAYER RELEVANCE PROPAGATION
fig, ax = plt.subplots(figsize=(10, 5))
cmap_l = plt.colormaps['tab10'].resampled(len(layer_names))

for i, (name, arr) in enumerate(layer_arrays.items()):
    abs_arr     = np.abs(arr)
    stage_means = [abs_arr[labels == s].mean() for s in unique_stages]
    ax.plot(
        unique_stages, stage_means, marker='o', linewidth=2,
        label=name.replace("_", " "), color=cmap_l(i)
    )

ax.set_xticks(unique_stages)
ax.set_xticklabels([f"Stage {s}" for s in unique_stages], fontsize=11)
ax.set_xlabel("Parkinson Stage", fontsize=11)
ax.set_ylabel("Mean Absolute Relevance", fontsize=11)
ax.set_title(
    "Cross-Layer Relevance Propagation Across Disease Stages\n"
    "(How much relevance each layer carries per stage)",
    fontsize=12, fontweight="bold"
)
ax.legend(title="Layer", fontsize=9)
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig(os.path.join(entropy_folder, "cross_layer_relevance_propagation.png"), dpi=180)
plt.close()


print("\nLRP analysis complete, results saved in:", OUTPUT_FOLDER)

d:\USTHB\M2\Stage Parkinson\Parkinson-prediction\.venv\Lib\site-packages\captum\attr\_core\lrp.py:207: UserWarning: Input Tensor 0 did not already require gradients, required_grads has been set automatically.
  gradient_mask = apply_gradient_requirements(input_tuple)


TypeError: Module of type <class 'torch.nn.modules.batchnorm.BatchNorm1d'> has no rule defined and nodefault rule exists for this module type. Please, set a ruleexplicitly for this module and assure that it is appropriatefor this type of layer.

## Version 03

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.init as init
from captum.attr import LRP, LayerLRP


# CONFIG
MODEL_PATH = "models/MedicalRecordProcessor_WEIGHTS_v2.pth"
CSV_PATH = "Dataset/CSV/paired_data_datscan.csv"

LABEL_COLUMN = "NHY"
ID_COLUMNS = ["PATNO", "EVENT_ID", "Image Data ID"]

TOP_N_GLOBAL = 10     # number of top global features to plot
TOP_N_PER_CLASS = 10  # number of top features per Parkinson stage
TOP_N_HEATMAP_FEATURES = 7 # number of features to show in heatmap 

OUTPUT_FOLDER = "lrp_outputs_g"


# MODEL DEFINITION
class MedicalRecordProcessor(nn.Module):
    def __init__(self, num_features=79, num_classes=5):
        super().__init__()
        self.num_features = num_features
        self.num_classes = num_classes
        
        self.mlp = nn.Sequential(
            nn.Linear(num_features, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4)
        )
        self.classifier = nn.Linear(512, num_classes)
        
        self._initialize_weights()
    
    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    init.constant_(m.bias, 0)
    
    def forward(self, x, classify=True):
        if x.shape[1] != self.num_features:
            raise ValueError(f"Expected {self.num_features} features, got {x.shape[1]}")
        
        features = self.mlp(x)
        if classify:
            return self.classifier(features)
        return features


# LOAD MODEL
device = torch.device("cpu")
model = MedicalRecordProcessor(num_features=79, num_classes=5)

checkpoint = torch.load(MODEL_PATH, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])

model.eval()


# LOAD DATA
df = pd.read_csv(CSV_PATH)

labels = df[LABEL_COLUMN].values.astype(int)

id_df = df[ID_COLUMNS]  # keep IDs to save later

features_df = df.drop(columns=[LABEL_COLUMN] + ID_COLUMNS, errors='ignore')
feature_names = features_df.columns.tolist()

X = features_df.values.astype(np.float32)
X_tensor = torch.tensor(X, dtype=torch.float32)


# LRP LAYER SETUP
layers_to_explain = [
    ("layer1_linear", model.mlp[0]),
    ("layer2_linear", model.mlp[3]),
    ("layer3_linear", model.mlp[6]),
    ("classifier", model.classifier)
]

layer_lrps = {name: LayerLRP(model, layer) for name, layer in layers_to_explain}
layer_outputs = {name: [] for name, _ in layers_to_explain}


# LRP COMPUTATION
lrp = LRP(model)

all_relevance = []

for i in range(X_tensor.shape[0]):
    x = X_tensor[i].unsqueeze(0) # adds batch dimension; (79,) -> (1, 79)
    # target_class = int(labels[i])
    pred_class = model(x).argmax(dim=1).item()

    relevance = lrp.attribute(x, target=pred_class)
    relevance = relevance.detach().cpu().numpy().flatten() # convert tensor to numpy array; (1, 79) -> (79,)

    # LRP relevance per layer
    for layer_name, layer_lrp in layer_lrps.items():
        layer_rel = layer_lrp.attribute(x, target=pred_class)
        layer_rel = layer_rel.detach().cpu().numpy().flatten()
        layer_outputs[layer_name].append(layer_rel)

    all_relevance.append(relevance)

all_relevance = np.array(all_relevance)


# save layer relevance
layer_folder = os.path.join(OUTPUT_FOLDER, "layer_relevance")
os.makedirs(layer_folder, exist_ok=True)

for layer_name, values in layer_outputs.items():
    layer_array = np.array(values)

    # create folder for this layer
    current_layer_folder = os.path.join(layer_folder, layer_name)
    os.makedirs(current_layer_folder, exist_ok=True)

    # dataframe per patient
    layer_df = pd.DataFrame(layer_array)

    # add patient IDs
    layer_df = pd.concat([id_df.reset_index(drop=True), layer_df], axis=1)

    layer_df.to_csv(
        os.path.join(current_layer_folder, "relevance_per_patient.csv"),
        index=False
    )

    # median relevance (global behaviour)
    median_relevance = np.median(layer_array, axis=0)

    median_df = pd.DataFrame({
        "neuron_index": range(len(median_relevance)),
        "median_relevance": median_relevance
    })

    median_df.to_csv(
        os.path.join(current_layer_folder, "median_relevance.csv"),
        index=False
    )


    # RELEVANCE FLOW ACROSS LAYERS
    flow_folder = os.path.join(OUTPUT_FOLDER, "relevance_flow")
    os.makedirs(flow_folder, exist_ok=True)

    flow_layers = ["input_features"]
    flow_values = [np.mean(np.abs(all_relevance))]

    for layer_name, values in layer_outputs.items():
        layer_array = np.array(values)
        flow_layers.append(layer_name)
        flow_values.append(np.mean(np.abs(layer_array)))

    flow_layers.append("prediction")

    # prediction relevance is always 1 for LRP explanation start
    flow_values.append(1.0)

    flow_df = pd.DataFrame({
        "layer": flow_layers,
        "mean_absolute_relevance": flow_values
    })

    flow_df.to_csv(os.path.join(flow_folder, "relevance_flow_across_layers.csv"), index=False)

    plt.figure(figsize=(8,5))
    plt.plot(flow_layers, flow_values, marker='o')
    plt.title("Relevance Flow Across Network Layers")
    plt.xlabel("Network Stage")
    plt.ylabel("Mean Absolute Relevance")
    plt.tight_layout()

    plt.savefig(os.path.join(flow_folder, "relevance_flow_across_layers.png"))
    plt.close()


# SAVE PER PATIENT
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

per_patient_df = pd.DataFrame(all_relevance, columns=feature_names)
per_patient_df = pd.concat([id_df.reset_index(drop=True), per_patient_df], axis=1)
per_patient_df[LABEL_COLUMN] = labels

per_patient_df.to_csv(f"{OUTPUT_FOLDER}/relevance_per_patient.csv", index=False)


# GLOBAL IMPORTANCE
global_mean_signed = all_relevance.mean(axis=0)
global_mean_abs = np.abs(all_relevance).mean(axis=0)

global_df = pd.DataFrame({
    "feature": feature_names,
    "mean_signed_relevance": global_mean_signed,
    "mean_abs_relevance": global_mean_abs
}).sort_values("mean_abs_relevance", ascending=False)


global_folder = os.path.join(OUTPUT_FOLDER, "global_importance")
os.makedirs(global_folder, exist_ok=True)

global_df.to_csv(f"{global_folder}/global_feature_importance.csv", index=False)


# PER CLASS IMPORTANCE
class_importance = {}

per_class_folder = os.path.join(OUTPUT_FOLDER, "class_importance")
os.makedirs(per_class_folder, exist_ok=True)

for c in sorted(np.unique(labels)):
    class_mask = labels == c
    class_rel = all_relevance[class_mask]

    mean_signed = class_rel.mean(axis=0)
    mean_abs = np.abs(class_rel).mean(axis=0)

    df_class = pd.DataFrame({
        "feature": feature_names,
        "mean_signed_relevance": mean_signed,
        "mean_abs_relevance": mean_abs
    }).sort_values("mean_abs_relevance", ascending=False)

    df_class.to_csv(f"{per_class_folder}/class_{c}_feature_importance.csv", index=False)
    class_importance[c] = df_class


# FEATURE RELEVANCE HEATMAP (Parkinson stage x top features)
# select top N features globally
top_features_indices = np.argsort(global_mean_abs)[-TOP_N_HEATMAP_FEATURES:]
top_feature_names = [feature_names[i] for i in top_features_indices]

stage_heatmap = []

for c in sorted(np.unique(labels)):
    class_mask = labels == c
    class_rel = np.abs(all_relevance[class_mask])

    # average relevance per feature for this stage (only top features)
    stage_mean = class_rel[:, top_features_indices].mean(axis=0)
    stage_heatmap.append(stage_mean)

stage_heatmap = np.array(stage_heatmap)

plt.figure(figsize=(14,6))

plt.imshow(
    stage_heatmap,
    aspect='auto',
    interpolation='nearest'
)

plt.colorbar(label="Mean Absolute LRP Relevance")

plt.yticks(
    range(len(np.unique(labels))),
    [f"Stage {c}" for c in sorted(np.unique(labels))]
)

plt.xticks(
    range(len(top_feature_names)),
    top_feature_names,
    rotation=90
)

plt.xlabel("Clinical Features")
plt.ylabel("Parkinson Stage")
plt.title(f"Top {TOP_N_HEATMAP_FEATURES} Feature Relevance Heatmap Across Parkinson Stages")

plt.tight_layout()

plt.savefig(os.path.join(OUTPUT_FOLDER, "feature_relevance_heatmap_top_features.png"))
plt.close()



# RELEVANCE PATHWAY DIAGRAM
# Compute mean absolute relevance for each "layer stage"
pathway_layers = ["Input Features"]
pathway_values = [np.mean(np.abs(all_relevance), axis=0).sum()]  # total relevance in inputs

# hidden layers
for layer_name, layer_vals in layer_outputs.items():
    layer_array = np.array(layer_vals)  # shape (num_patients, num_neurons)
    mean_abs_per_neuron = np.mean(np.abs(layer_array), axis=0)
    pathway_layers.append(layer_name)
    pathway_values.append(mean_abs_per_neuron.sum())  # sum relevance per layer

# prediction
pathway_layers.append("Prediction")
pathway_values.append(1.0)  # LRP assigns total relevance = 1 at prediction

# Plot the pathway
plt.figure(figsize=(10,6))
plt.plot(pathway_layers, pathway_values, marker='o', linestyle='-', color='blue')
plt.title("Feature → Hidden Neurons → Prediction Relevance Pathway")
plt.ylabel("Sum of Absolute Relevance")
plt.xlabel("Network Stage")
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()

# save figure
pathway_folder = os.path.join(OUTPUT_FOLDER, "relevance_pathway")
os.makedirs(pathway_folder, exist_ok=True)
plt.savefig(os.path.join(pathway_folder, "relevance_pathway.png"))
plt.close()


# STAGE PROGRESSION IMPORTANCE CURVE
unique_stages = sorted(np.unique(labels))  # e.g., [0,1,2,3,4]

# store mean relevance per stage for all features
stage_feature_importance = np.zeros((len(unique_stages), len(feature_names)))

for i, stage in enumerate(unique_stages):
    mask = labels == stage
    stage_feature_importance[i] = np.abs(all_relevance[mask]).mean(axis=0)

# select top features globally
top_features_indices = np.argsort(global_mean_abs)[-TOP_N_HEATMAP_FEATURES:]
top_feature_names = [feature_names[i] for i in top_features_indices]

stage_feature_importance_top = stage_feature_importance[:, top_features_indices]

plt.figure(figsize=(14,6))

for i, feature_name in enumerate(top_feature_names):
    plt.plot(
        unique_stages,
        stage_feature_importance_top[:, i],
        marker='o',
        label=feature_name
    )

plt.xticks(unique_stages, [f"Stage {s}" for s in unique_stages])
plt.xlabel("Parkinson Stage")
plt.ylabel("Mean Absolute Feature Relevance")
plt.title(f"Stage Progression Importance Curve (Top {TOP_N_HEATMAP_FEATURES} Features)")
plt.legend(bbox_to_anchor=(1.05,1), loc='upper left', fontsize=8)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()

# save figure
curve_folder = os.path.join(OUTPUT_FOLDER, "stage_progression")
os.makedirs(curve_folder, exist_ok=True)
plt.savefig(os.path.join(curve_folder, "stage_progression_importance_curve.png"))
plt.close()


# PLOTTING
def plot_top_features(df_sorted, n, title, save_path):
    top_df = df_sorted.head(n)
    plt.figure(figsize=(8, 5))
    plt.barh(top_df["feature"], top_df["mean_abs_relevance"])
    plt.gca().invert_yaxis()
    plt.title(title)
    plt.xlabel("Mean Absolute Relevance")
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()


# global plot
plot_top_features(
    global_df,
    TOP_N_GLOBAL,
    f"Top {TOP_N_GLOBAL} Global Important Features",
    f"{global_folder}/top_global_features.png"
)

# per class plots
for c, df_class in class_importance.items():
    plot_top_features(
        df_class,
        TOP_N_PER_CLASS,
        f"Top {TOP_N_PER_CLASS} Features - Parkinson Stage {c}",
        f"{per_class_folder}/top_features_class_{c}.png"
    )

print("\nLRP analysis complete, results saved in:", OUTPUT_FOLDER)

## Version 04

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.colors import TwoSlopeNorm
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.init as init

from captum.attr import LRP, LayerLRP
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from scipy.stats import spearmanr
from scipy.cluster.hierarchy import dendrogram, linkage

# ─────────────────────────── CONFIG ───────────────────────────
MODEL_PATH    = "models/MedicalRecordProcessor_WEIGHTS_v2.pth"
CSV_PATH      = "Dataset/CSV/paired_data_datscan.csv"

LABEL_COLUMN  = "NHY"
ID_COLUMNS    = ["PATNO", "EVENT_ID", "Image Data ID"]

TOP_N_GLOBAL          = 10
TOP_N_PER_CLASS       = 10
TOP_N_HEATMAP_FEATURES = 7

OUTPUT_FOLDER = "lrp_outputs_c"

STAGE_COLORS = {0: "#4C72B0", 1: "#DD8452", 2: "#55A868", 3: "#C44E52", 4: "#8172B2"}
STAGE_PALETTE = list(STAGE_COLORS.values())


# ─────────────────────────── MODEL ────────────────────────────
class MedicalRecordProcessor(nn.Module):
    def __init__(self, num_features=79, num_classes=5):
        super().__init__()
        self.num_features = num_features
        self.num_classes  = num_classes
        self.mlp = nn.Sequential(
            nn.Linear(num_features, 128), nn.ReLU(inplace=True), nn.Dropout(0.3),
            nn.Linear(128, 256),          nn.ReLU(inplace=True), nn.Dropout(0.3),
            nn.Linear(256, 512),          nn.ReLU(inplace=True), nn.Dropout(0.4),
        )
        self.classifier = nn.Linear(512, num_classes)
        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None: init.constant_(m.bias, 0)

    def forward(self, x, classify=True):
        if x.shape[1] != self.num_features:
            raise ValueError(f"Expected {self.num_features} features, got {x.shape[1]}")
        features = self.mlp(x)
        return self.classifier(features) if classify else features


# ─────────────────────────── LOAD ─────────────────────────────
device = torch.device("cpu")
model  = MedicalRecordProcessor(num_features=79, num_classes=5)
checkpoint = torch.load(MODEL_PATH, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

df           = pd.read_csv(CSV_PATH)
labels       = df[LABEL_COLUMN].values.astype(int)
id_df        = df[ID_COLUMNS]
features_df  = df.drop(columns=[LABEL_COLUMN] + ID_COLUMNS, errors='ignore')
feature_names = features_df.columns.tolist()
X            = features_df.values.astype(np.float32)
X_tensor     = torch.tensor(X, dtype=torch.float32)
unique_stages = sorted(np.unique(labels))


# ─────────────────────────── LRP SETUP ────────────────────────
layers_to_explain = [
    ("layer1_linear", model.mlp[0]),
    ("layer2_linear", model.mlp[3]),
    ("layer3_linear", model.mlp[6]),
    ("classifier",    model.classifier),
]
layer_lrps    = {name: LayerLRP(model, layer) for name, layer in layers_to_explain}
layer_outputs = {name: [] for name, _ in layers_to_explain}
lrp           = LRP(model)

all_relevance = []
pred_classes  = []

print("Computing LRP attributions …")
for i in range(X_tensor.shape[0]):
    x          = X_tensor[i].unsqueeze(0)
    pred_class = model(x).argmax(dim=1).item()
    pred_classes.append(pred_class)

    relevance  = lrp.attribute(x, target=pred_class)
    relevance  = relevance.detach().cpu().numpy().flatten()
    all_relevance.append(relevance)

    for layer_name, layer_lrp in layer_lrps.items():
        layer_rel = layer_lrp.attribute(x, target=pred_class)
        layer_outputs[layer_name].append(layer_rel.detach().cpu().numpy().flatten())

all_relevance = np.array(all_relevance)
pred_classes  = np.array(pred_classes)
correct_mask  = pred_classes == labels

print(f"Overall accuracy: {correct_mask.mean():.3f}")

global_mean_signed = all_relevance.mean(axis=0)
global_mean_abs    = np.abs(all_relevance).mean(axis=0)
global_df = pd.DataFrame({
    "feature":               feature_names,
    "mean_signed_relevance": global_mean_signed,
    "mean_abs_relevance":    global_mean_abs,
}).sort_values("mean_abs_relevance", ascending=False)

os.makedirs(OUTPUT_FOLDER, exist_ok=True)


# ══════════════════════════════════════════════════════════════
# 1. GLOBAL TOP-N BAR CHART  (signed, diverging)
# ══════════════════════════════════════════════════════════════
def plot_signed_global(global_df, n, out_path):
    top = global_df.head(n).copy()
    colors = ["#C44E52" if v > 0 else "#4C72B0" for v in top["mean_signed_relevance"]]
    fig, ax = plt.subplots(figsize=(9, 6))
    bars = ax.barh(top["feature"], top["mean_signed_relevance"], color=colors, edgecolor="white", linewidth=0.5)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlabel("Mean Signed LRP Relevance", fontsize=11)
    ax.set_title(f"Top {n} Features — Signed Global Relevance", fontsize=13, fontweight="bold")
    ax.invert_yaxis()
    pos_patch = mpatches.Patch(color="#C44E52", label="Positive (promotes prediction)")
    neg_patch = mpatches.Patch(color="#4C72B0", label="Negative (suppresses prediction)")
    ax.legend(handles=[pos_patch, neg_patch], fontsize=9)
    plt.tight_layout()
    plt.savefig(out_path, dpi=180)
    plt.close()
    print(f"  Saved: {out_path}")

global_folder = os.path.join(OUTPUT_FOLDER, "global_importance")
os.makedirs(global_folder, exist_ok=True)
global_df.to_csv(f"{global_folder}/global_feature_importance.csv", index=False)
plot_signed_global(global_df, TOP_N_GLOBAL, f"{global_folder}/top_global_features_signed.png")


# ══════════════════════════════════════════════════════════════
# 2. PER-CLASS SIGNED DIVERGING HEATMAP  (thesis-quality)
# ══════════════════════════════════════════════════════════════
top_idx   = np.argsort(global_mean_abs)[-TOP_N_HEATMAP_FEATURES:]
top_names = [feature_names[i] for i in top_idx]

signed_heatmap = np.zeros((len(unique_stages), len(top_idx)))
abs_heatmap    = np.zeros_like(signed_heatmap)

for i, stage in enumerate(unique_stages):
    mask = labels == stage
    signed_heatmap[i] = all_relevance[mask][:, top_idx].mean(axis=0)
    abs_heatmap[i]    = np.abs(all_relevance[mask])[:, top_idx].mean(axis=0)

heatmap_folder = os.path.join(OUTPUT_FOLDER, "heatmaps")
os.makedirs(heatmap_folder, exist_ok=True)

# Signed diverging heatmap
fig, ax = plt.subplots(figsize=(14, 5))
vmax = np.abs(signed_heatmap).max()
norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
im   = ax.imshow(signed_heatmap, aspect='auto', cmap='RdBu_r', norm=norm, interpolation='nearest')
plt.colorbar(im, ax=ax, label="Mean Signed LRP Relevance")
ax.set_yticks(range(len(unique_stages)))
ax.set_yticklabels([f"Stage {c}" for c in unique_stages], fontsize=11)
ax.set_xticks(range(len(top_names)))
ax.set_xticklabels(top_names, rotation=45, ha='right', fontsize=10)
ax.set_title(f"Signed Feature Relevance Heatmap — Top {TOP_N_HEATMAP_FEATURES} Features × Parkinson Stage",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(heatmap_folder, "signed_relevance_heatmap.png"), dpi=180)
plt.close()
print(f"  Saved: signed_relevance_heatmap.png")

# Absolute heatmap with hierarchical clustering on features
row_link = linkage(abs_heatmap.T, method='ward')
from scipy.cluster.hierarchy import leaves_list
order = leaves_list(row_link)

fig, axes = plt.subplots(1, 2, figsize=(16, 5),
                          gridspec_kw={"width_ratios": [1, 4]})
dn = dendrogram(row_link, ax=axes[0], orientation='left', no_labels=True,
                color_threshold=0, above_threshold_color='gray')
axes[0].axis('off')
im = axes[1].imshow(abs_heatmap[:, order], aspect='auto', cmap='YlOrRd', interpolation='nearest')
plt.colorbar(im, ax=axes[1], label="Mean Absolute LRP Relevance")
axes[1].set_yticks(range(len(unique_stages)))
axes[1].set_yticklabels([f"Stage {c}" for c in unique_stages], fontsize=11)
axes[1].set_xticks(range(len(top_names)))
axes[1].set_xticklabels([top_names[i] for i in order], rotation=45, ha='right', fontsize=10)
axes[1].set_title(f"Clustered Absolute Feature Relevance Heatmap", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(heatmap_folder, "clustered_abs_relevance_heatmap.png"), dpi=180)
plt.close()
print(f"  Saved: clustered_abs_relevance_heatmap.png")


# ══════════════════════════════════════════════════════════════
# 3. CORRECT vs MISCLASSIFIED RELEVANCE COMPARISON
# ══════════════════════════════════════════════════════════════
correct_folder = os.path.join(OUTPUT_FOLDER, "correct_vs_misclassified")
os.makedirs(correct_folder, exist_ok=True)

correct_rel   = np.abs(all_relevance[correct_mask]).mean(axis=0)
incorrect_rel = np.abs(all_relevance[~correct_mask]).mean(axis=0)

# Difference: positive = more relevant in correct predictions
diff_rel = correct_rel - incorrect_rel
top_diff_idx = np.argsort(np.abs(diff_rel))[-TOP_N_GLOBAL:]
top_diff_names = [feature_names[i] for i in top_diff_idx]

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
bar_colors = ["#55A868" if d > 0 else "#C44E52" for d in diff_rel[top_diff_idx]]

axes[0].barh(top_diff_names, correct_rel[top_diff_idx],   color="#55A868", label="Correct")
axes[0].barh(top_diff_names, -incorrect_rel[top_diff_idx], color="#C44E52", label="Misclassified")
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].set_xlabel("Mean Absolute Relevance (←Misclassified | Correct→)")
axes[0].set_title("Feature Relevance: Correct vs Misclassified", fontweight="bold")
axes[0].legend()
axes[0].invert_yaxis()

axes[1].barh(top_diff_names, diff_rel[top_diff_idx], color=bar_colors)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_xlabel("Δ Relevance (Correct − Misclassified)")
axes[1].set_title("Relevance Difference", fontweight="bold")

plt.suptitle(
    f"LRP Relevance: Correctly vs Incorrectly Classified Patients\n"
    f"(n_correct={correct_mask.sum()}, n_misclassified={(~correct_mask).sum()})",
    fontsize=12, fontweight="bold"
)
plt.tight_layout()
plt.savefig(os.path.join(correct_folder, "correct_vs_misclassified_relevance.png"), dpi=180)
plt.close()
print(f"  Saved: correct_vs_misclassified_relevance.png")

pd.DataFrame({
    "feature":        feature_names,
    "correct_rel":    correct_rel,
    "incorrect_rel":  incorrect_rel,
    "delta":          diff_rel,
}).sort_values("delta", ascending=False).to_csv(
    os.path.join(correct_folder, "correct_vs_misclassified_relevance.csv"), index=False
)


# ══════════════════════════════════════════════════════════════
# 4. PATIENT STABILITY — INTRA-STAGE RELEVANCE STD HEATMAP
# ══════════════════════════════════════════════════════════════
stability_folder = os.path.join(OUTPUT_FOLDER, "relevance_stability")
os.makedirs(stability_folder, exist_ok=True)

std_heatmap  = np.zeros((len(unique_stages), len(top_idx)))
mean_heatmap = np.zeros_like(std_heatmap)

for i, stage in enumerate(unique_stages):
    mask = labels == stage
    stage_rel = np.abs(all_relevance[mask])[:, top_idx]
    mean_heatmap[i] = stage_rel.mean(axis=0)
    std_heatmap[i]  = stage_rel.std(axis=0)

# Coefficient of variation (std / mean) — scale-free instability measure
cv_heatmap = np.divide(std_heatmap, mean_heatmap + 1e-8)

fig, axes = plt.subplots(1, 2, figsize=(18, 5))
for ax, data, title, cmap in zip(
    axes,
    [mean_heatmap, cv_heatmap],
    ["Mean Absolute Relevance", "Coefficient of Variation (Instability)"],
    ["YlOrRd", "Blues"],
):
    im = ax.imshow(data, aspect='auto', cmap=cmap, interpolation='nearest')
    plt.colorbar(im, ax=ax)
    ax.set_yticks(range(len(unique_stages)))
    ax.set_yticklabels([f"Stage {c}" for c in unique_stages])
    ax.set_xticks(range(len(top_names)))
    ax.set_xticklabels(top_names, rotation=45, ha='right', fontsize=9)
    ax.set_title(title, fontsize=12, fontweight="bold")

plt.suptitle("Relevance Mean vs Intra-Stage Stability (Top Features)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(stability_folder, "relevance_stability_heatmap.png"), dpi=180)
plt.close()
print(f"  Saved: relevance_stability_heatmap.png")


# ══════════════════════════════════════════════════════════════
# 5. PCA / t-SNE OF LRP RELEVANCE VECTORS
# ══════════════════════════════════════════════════════════════
embedding_folder = os.path.join(OUTPUT_FOLDER, "embeddings")
os.makedirs(embedding_folder, exist_ok=True)

colors_per_patient = [STAGE_COLORS[s] for s in labels]
stage_handles = [mpatches.Patch(color=STAGE_COLORS[s], label=f"Stage {s}") for s in unique_stages]

# PCA
pca        = PCA(n_components=2, random_state=42)
pca_coords = pca.fit_transform(all_relevance)

fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(pca_coords[:, 0], pca_coords[:, 1],
                     c=colors_per_patient, alpha=0.7, s=25, linewidths=0)
ax.legend(handles=stage_handles, title="Parkinson Stage", fontsize=9)
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)", fontsize=11)
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)", fontsize=11)
ax.set_title("PCA of LRP Relevance Vectors per Patient", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(embedding_folder, "pca_relevance.png"), dpi=180)
plt.close()
print(f"  Saved: pca_relevance.png")

# t-SNE
tsne        = TSNE(n_components=2, perplexity=30, random_state=42, max_iter=1000)
tsne_coords = tsne.fit_transform(all_relevance)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# t-SNE colored by TRUE label
axes[0].scatter(tsne_coords[:, 0], tsne_coords[:, 1],
                c=colors_per_patient, alpha=0.7, s=25, linewidths=0)
axes[0].legend(handles=stage_handles, title="True Stage", fontsize=9)
axes[0].set_title("t-SNE of LRP Relevance Vectors\n(True Label)", fontsize=12, fontweight="bold")
axes[0].set_xlabel("t-SNE 1"); axes[0].set_ylabel("t-SNE 2")

# t-SNE colored by PREDICTED label
colors_pred = [STAGE_COLORS.get(p, "#999999") for p in pred_classes]
pred_handles = [mpatches.Patch(color=STAGE_COLORS[s], label=f"Pred Stage {s}") for s in unique_stages]
axes[1].scatter(tsne_coords[:, 0], tsne_coords[:, 1],
                c=colors_pred, alpha=0.7, s=25, linewidths=0)
# Mark misclassified with black ring
axes[1].scatter(tsne_coords[~correct_mask, 0], tsne_coords[~correct_mask, 1],
                facecolors='none', edgecolors='black', s=50, linewidths=0.8, label="Misclassified")
axes[1].legend(handles=pred_handles + [mpatches.Patch(facecolor='none', edgecolor='black', label="Misclassified")],
               title="Predicted Stage", fontsize=9)
axes[1].set_title("t-SNE of LRP Relevance Vectors\n(Predicted Label + Errors)", fontsize=12, fontweight="bold")
axes[1].set_xlabel("t-SNE 1"); axes[1].set_ylabel("t-SNE 2")

plt.suptitle("LRP Relevance Space — Patient Clustering", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(embedding_folder, "tsne_relevance.png"), dpi=180)
plt.close()
print(f"  Saved: tsne_relevance.png")


# ══════════════════════════════════════════════════════════════
# 6. FEATURE CORRELATION vs RELEVANCE RANK (methodological plot)
# ══════════════════════════════════════════════════════════════
corr_folder = os.path.join(OUTPUT_FOLDER, "correlation_analysis")
os.makedirs(corr_folder, exist_ok=True)

# Pairwise Spearman correlation between feature values (mean per feature pair)
feat_corr_matrix = np.corrcoef(X.T)  # (79 x 79) Pearson for speed

relevance_rank = np.argsort(np.argsort(global_mean_abs))  # rank 0=least important
top_k = 20
top_k_idx = np.argsort(global_mean_abs)[-top_k:]

sub_corr = np.abs(feat_corr_matrix[np.ix_(top_k_idx, top_k_idx)])
sub_names = [feature_names[i] for i in top_k_idx]

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(sub_corr, dtype=bool))
sns.heatmap(sub_corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            xticklabels=sub_names, yticklabels=sub_names,
            ax=ax, linewidths=0.5, vmin=0, vmax=1, annot_kws={"size": 7})
ax.set_title(f"Feature Correlation Matrix — Top {top_k} Most Relevant Features",
             fontsize=13, fontweight="bold")
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(corr_folder, "feature_correlation_top_relevant.png"), dpi=180)
plt.close()
print(f"  Saved: feature_correlation_top_relevant.png")

# Scatter: feature variance vs mean absolute relevance (do variable features get more credit?)
feat_variance  = X.var(axis=0)
fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(feat_variance, global_mean_abs, alpha=0.6, s=20,
                c=relevance_rank, cmap="viridis")
plt.colorbar(sc, ax=ax, label="Relevance rank")
ax.set_xlabel("Feature Variance (in dataset)", fontsize=11)
ax.set_ylabel("Mean Absolute LRP Relevance", fontsize=11)
ax.set_title("Feature Variance vs LRP Relevance\n(Do high-variance features get more credit?)",
             fontsize=12, fontweight="bold")
rho, p = spearmanr(feat_variance, global_mean_abs)
ax.text(0.05, 0.92, f"Spearman ρ = {rho:.3f}  (p={p:.3f})",
        transform=ax.transAxes, fontsize=10, color="black",
        bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray"))
plt.tight_layout()
plt.savefig(os.path.join(corr_folder, "variance_vs_relevance.png"), dpi=180)
plt.close()
print(f"  Saved: variance_vs_relevance.png")


# ══════════════════════════════════════════════════════════════
# 7. RELEVANCE FLOW ACROSS LAYERS  (existing, but improved)
# ══════════════════════════════════════════════════════════════
flow_folder = os.path.join(OUTPUT_FOLDER, "relevance_flow")
os.makedirs(flow_folder, exist_ok=True)

flow_layers = ["Input\nFeatures"]
flow_values = [np.mean(np.abs(all_relevance))]

for layer_name, vals in layer_outputs.items():
    la = np.array(vals)
    flow_layers.append(layer_name.replace("_", "\n"))
    flow_values.append(np.mean(np.abs(la)))

flow_layers.append("Prediction")
flow_values.append(1.0)

fig, ax = plt.subplots(figsize=(10, 5))
x_pos = range(len(flow_layers))
ax.plot(x_pos, flow_values, marker='o', linewidth=2, color="#4C72B0", markersize=9, zorder=3)
ax.fill_between(x_pos, flow_values, alpha=0.15, color="#4C72B0")
for x, y, label in zip(x_pos, flow_values, flow_layers):
    ax.annotate(f"{y:.4f}", (x, y), textcoords="offset points", xytext=(0, 12),
                ha='center', fontsize=9)
ax.set_xticks(x_pos)
ax.set_xticklabels(flow_layers, fontsize=10)
ax.set_ylabel("Mean Absolute Relevance", fontsize=11)
ax.set_title("LRP Relevance Flow Across Network Layers", fontsize=13, fontweight="bold")
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig(os.path.join(flow_folder, "relevance_flow_across_layers.png"), dpi=180)
plt.close()

pd.DataFrame({"layer": flow_layers, "mean_absolute_relevance": flow_values}).to_csv(
    os.path.join(flow_folder, "relevance_flow_across_layers.csv"), index=False
)
print(f"  Saved: relevance_flow_across_layers.png")


# ══════════════════════════════════════════════════════════════
# 8. STAGE PROGRESSION IMPORTANCE CURVE  (existing, improved)
# ══════════════════════════════════════════════════════════════
curve_folder = os.path.join(OUTPUT_FOLDER, "stage_progression")
os.makedirs(curve_folder, exist_ok=True)

stage_feat_imp = np.zeros((len(unique_stages), len(feature_names)))
for i, stage in enumerate(unique_stages):
    mask = labels == stage
    stage_feat_imp[i] = np.abs(all_relevance[mask]).mean(axis=0)

top_idx2  = np.argsort(global_mean_abs)[-TOP_N_HEATMAP_FEATURES:]
top_names2 = [feature_names[i] for i in top_idx2]
stage_imp_top = stage_feat_imp[:, top_idx2]

fig, ax = plt.subplots(figsize=(12, 6))
cmap_prog = plt.cm.get_cmap('tab10', len(top_names2))
for i, feat_name in enumerate(top_names2):
    ax.plot(unique_stages, stage_imp_top[:, i],
            marker='o', label=feat_name, color=cmap_prog(i), linewidth=2)
ax.set_xticks(unique_stages)
ax.set_xticklabels([f"Stage {s}" for s in unique_stages], fontsize=11)
ax.set_xlabel("Parkinson Stage (H&Y)", fontsize=11)
ax.set_ylabel("Mean Absolute LRP Relevance", fontsize=11)
ax.set_title(f"Feature Relevance Across Disease Progression\n(Top {TOP_N_HEATMAP_FEATURES} Features)",
             fontsize=13, fontweight="bold")
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig(os.path.join(curve_folder, "stage_progression_importance_curve.png"), dpi=180)
plt.close()
print(f"  Saved: stage_progression_importance_curve.png")


# ══════════════════════════════════════════════════════════════
# 9. PER-CLASS SIGNED DIVERGING BAR CHARTS  (small multiples)
# ══════════════════════════════════════════════════════════════
per_class_folder = os.path.join(OUTPUT_FOLDER, "class_importance")
os.makedirs(per_class_folder, exist_ok=True)

n_stages = len(unique_stages)
fig, axes = plt.subplots(1, n_stages, figsize=(5 * n_stages, 7), sharey=False)
if n_stages == 1: axes = [axes]

for ax, stage in zip(axes, unique_stages):
    mask      = labels == stage
    class_rel = all_relevance[mask]
    signed    = class_rel.mean(axis=0)
    top_idx_c = np.argsort(np.abs(signed))[-TOP_N_PER_CLASS:]
    feat_sub  = [feature_names[i] for i in top_idx_c]
    vals_sub  = signed[top_idx_c]
    bar_colors = ["#C44E52" if v > 0 else "#4C72B0" for v in vals_sub]

    ax.barh(feat_sub, vals_sub, color=bar_colors, edgecolor='white', linewidth=0.4)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(f"Stage {stage}\n(n={mask.sum()})", fontsize=11, fontweight="bold",
                 color=STAGE_COLORS[stage])
    ax.set_xlabel("Mean Signed Relevance", fontsize=9)
    ax.invert_yaxis()
    ax.tick_params(axis='y', labelsize=8)

plt.suptitle(f"Top {TOP_N_PER_CLASS} Feature Relevance Per Parkinson Stage (Signed)",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(per_class_folder, "per_stage_signed_relevance.png"), dpi=180)
plt.close()
print(f"  Saved: per_stage_signed_relevance.png")

# Save per-class CSVs
for stage in unique_stages:
    mask    = labels == stage
    rel_sub = all_relevance[mask]
    df_c = pd.DataFrame({
        "feature":               feature_names,
        "mean_signed_relevance": rel_sub.mean(axis=0),
        "mean_abs_relevance":    np.abs(rel_sub).mean(axis=0),
    }).sort_values("mean_abs_relevance", ascending=False)
    df_c.to_csv(f"{per_class_folder}/class_{stage}_feature_importance.csv", index=False)


# ══════════════════════════════════════════════════════════════
# 10. SAVE MASTER PER-PATIENT CSV
# ══════════════════════════════════════════════════════════════
per_patient_df = pd.DataFrame(all_relevance, columns=feature_names)
per_patient_df = pd.concat([id_df.reset_index(drop=True), per_patient_df], axis=1)
per_patient_df[LABEL_COLUMN]    = labels
per_patient_df["predicted_class"] = pred_classes
per_patient_df["correct"]         = correct_mask
per_patient_df.to_csv(f"{OUTPUT_FOLDER}/relevance_per_patient.csv", index=False)


# ══════════════════════════════════════════════════════════════
# 11. LAYER RELEVANCE CSVs  (existing logic, kept)
# ══════════════════════════════════════════════════════════════
layer_folder = os.path.join(OUTPUT_FOLDER, "layer_relevance")
os.makedirs(layer_folder, exist_ok=True)

for layer_name, values in layer_outputs.items():
    layer_array = np.array(values)
    cur = os.path.join(layer_folder, layer_name)
    os.makedirs(cur, exist_ok=True)
    layer_df = pd.DataFrame(layer_array)
    layer_df = pd.concat([id_df.reset_index(drop=True), layer_df], axis=1)
    layer_df.to_csv(os.path.join(cur, "relevance_per_patient.csv"), index=False)
    median_df = pd.DataFrame({
        "neuron_index":    range(layer_array.shape[1]),
        "median_relevance": np.median(layer_array, axis=0),
    })
    median_df.to_csv(os.path.join(cur, "median_relevance.csv"), index=False)

# ══════════════════════════════════════════════════════════════
# LAYER RELEVANCE PLOTS  — add this block after the layer
# relevance CSVs are saved (after the layer_outputs loop)
# ══════════════════════════════════════════════════════════════

from scipy.stats import entropy as scipy_entropy

layer_plot_folder = os.path.join(OUTPUT_FOLDER, "layer_relevance", "plots")
os.makedirs(layer_plot_folder, exist_ok=True)

# Collect arrays once
layer_arrays = {name: np.array(vals) for name, vals in layer_outputs.items()}
layer_names  = list(layer_arrays.keys())


# ──────────────────────────────────────────────────────────────
# PLOT 1: Neuron Relevance Distribution per Layer (violin plot)
# Shows how broadly or narrowly relevance is spread across neurons
# ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, len(layer_names), figsize=(5 * len(layer_names), 6), sharey=False)
if len(layer_names) == 1:
    axes = [axes]

for ax, (name, arr) in zip(axes, layer_arrays.items()):
    # arr shape: (n_patients, n_neurons)
    abs_arr = np.abs(arr)

    # mean absolute relevance per neuron (across patients)
    neuron_means = abs_arr.mean(axis=0)  # shape: (n_neurons,)

    parts = ax.violinplot(
        neuron_means,
        positions=[0],
        showmedians=True,
        showextrema=True,
    )
    for pc in parts['bodies']:
        pc.set_facecolor("#4C72B0")
        pc.set_alpha(0.7)
    parts['cmedians'].set_color("#C44E52")
    parts['cmedians'].set_linewidth(2)

    ax.set_xticks([0])
    ax.set_xticklabels([name.replace("_", "\n")], fontsize=10)
    ax.set_ylabel("Mean Abs. Neuron Relevance", fontsize=9)
    ax.set_title(f"{name}\n({arr.shape[1]} neurons)", fontsize=10, fontweight="bold")
    ax.grid(True, linestyle='--', alpha=0.4)

plt.suptitle("Neuron Relevance Distribution Across Layers\n(each violin = distribution over neurons)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(layer_plot_folder, "neuron_relevance_distribution.png"), dpi=180)
plt.close()
print(f"  Saved: neuron_relevance_distribution.png")


# ──────────────────────────────────────────────────────────────
# PLOT 2: Top-N Neurons Heatmap per Layer × Parkinson Stage
# Shows which specific neurons are most active per disease stage
# ──────────────────────────────────────────────────────────────
TOP_N_NEURONS = 20  # how many top neurons to show per layer (capped by layer size)

for name, arr in layer_arrays.items():
    abs_arr = np.abs(arr)  # (n_patients, n_neurons)

    # cap to actual number of neurons in this layer
    n_neurons_actual = arr.shape[1]
    top_n = min(TOP_N_NEURONS, n_neurons_actual)

    # pick top neurons by global mean relevance
    global_neuron_mean = abs_arr.mean(axis=0)
    top_neuron_idx     = np.argsort(global_neuron_mean)[-top_n:]
    top_neuron_labels  = [f"N{i}" for i in top_neuron_idx]

    # mean relevance per stage for those top neurons
    stage_neuron_matrix = np.zeros((len(unique_stages), top_n))
    for i, stage in enumerate(unique_stages):
        mask = labels == stage
        stage_neuron_matrix[i] = abs_arr[mask][:, top_neuron_idx].mean(axis=0)

    fig, ax = plt.subplots(figsize=(14, 4))
    im = ax.imshow(stage_neuron_matrix, aspect='auto', cmap='YlOrRd', interpolation='nearest')
    plt.colorbar(im, ax=ax, label="Mean Abs. Relevance")
    ax.set_yticks(range(len(unique_stages)))
    ax.set_yticklabels([f"Stage {s}" for s in unique_stages], fontsize=10)
    ax.set_xticks(range(top_n))
    ax.set_xticklabels(top_neuron_labels, rotation=45, ha='right', fontsize=8)
    ax.set_xlabel("Neuron Index (top by global relevance)", fontsize=10)
    ax.set_ylabel("Parkinson Stage", fontsize=10)
    ax.set_title(f"Top {top_n} Neuron Relevance per Stage — {name}",
                 fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(layer_plot_folder, f"neuron_heatmap_{name}.png"), dpi=180)
    plt.close()
    print(f"  Saved: neuron_heatmap_{name}.png")


# ──────────────────────────────────────────────────────────────
# PLOT 3: Layer Specialization — Relevance Entropy per Layer
# Low entropy = relevance concentrated in few neurons (specialized)
# High entropy = spread evenly (distributed representation)
# ──────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))

entropy_per_stage = {name: [] for name in layer_names}

for name, arr in layer_arrays.items():
    abs_arr = np.abs(arr)
    for stage in unique_stages:
        mask = labels == stage
        stage_mean = abs_arr[mask].mean(axis=0)
        # normalize to probability distribution, then compute entropy
        prob = stage_mean / (stage_mean.sum() + 1e-8)
        ent  = scipy_entropy(prob + 1e-10)
        entropy_per_stage[name].append(ent)

x     = np.arange(len(unique_stages))
width = 0.8 / len(layer_names)
cmap_layers = plt.cm.get_cmap('Set2', len(layer_names))

for i, name in enumerate(layer_names):
    offset = (i - len(layer_names) / 2 + 0.5) * width
    ax.bar(x + offset, entropy_per_stage[name], width=width,
           label=name.replace("_", " "), color=cmap_layers(i), edgecolor='white', linewidth=0.5)

ax.set_xticks(x)
ax.set_xticklabels([f"Stage {s}" for s in unique_stages], fontsize=11)
ax.set_xlabel("Parkinson Stage", fontsize=11)
ax.set_ylabel("Relevance Entropy (nats)", fontsize=11)
ax.set_title("Layer Specialization — Relevance Entropy per Layer per Stage\n"
             "(Low entropy = few neurons dominate | High entropy = distributed)",
             fontsize=12, fontweight="bold")
ax.legend(title="Layer", fontsize=9)
ax.grid(True, axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig(os.path.join(layer_plot_folder, "layer_specialization_entropy.png"), dpi=180)
plt.close()
print(f"  Saved: layer_specialization_entropy.png")


# ──────────────────────────────────────────────────────────────
# PLOT 4: Cross-layer Relevance Propagation Summary
# One line per layer showing mean abs relevance across stages —
# lets you see which layer "amplifies" signal for which stage
# ──────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
cmap_l = plt.cm.get_cmap('tab10', len(layer_names))

for i, (name, arr) in enumerate(layer_arrays.items()):
    abs_arr = np.abs(arr)
    stage_means = [abs_arr[labels == s].mean() for s in unique_stages]
    ax.plot(unique_stages, stage_means, marker='o', linewidth=2,
            label=name.replace("_", " "), color=cmap_l(i))

ax.set_xticks(unique_stages)
ax.set_xticklabels([f"Stage {s}" for s in unique_stages], fontsize=11)
ax.set_xlabel("Parkinson Stage", fontsize=11)
ax.set_ylabel("Mean Absolute Relevance", fontsize=11)
ax.set_title("Cross-Layer Relevance Propagation Across Disease Stages\n"
             "(How much relevance each layer carries per stage)",
             fontsize=12, fontweight="bold")
ax.legend(title="Layer", fontsize=9)
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig(os.path.join(layer_plot_folder, "cross_layer_relevance_propagation.png"), dpi=180)
plt.close()
print(f"  Saved: cross_layer_relevance_propagation.png")

print(f"\n✓ Layer relevance plots saved to: {layer_plot_folder}")


print("\n✓ All thesis outputs saved to:", OUTPUT_FOLDER)
print("\nOutput summary:")
print("  1. global_importance/       — signed global bar chart + CSV")
print("  2. heatmaps/                — signed diverging + clustered heatmaps")
print("  3. correct_vs_misclassified/ — relevance comparison correct/wrong patients")
print("  4. relevance_stability/     — mean + coefficient-of-variation heatmaps")
print("  5. embeddings/              — PCA + t-SNE of relevance vectors")
print("  6. correlation_analysis/    — feature correlation + variance-vs-relevance")
print("  7. relevance_flow/          — layer-by-layer flow curve")
print("  8. stage_progression/       — progression curve top features")
print("  9. class_importance/        — per-stage signed bar charts + CSVs")
print(" 10. relevance_per_patient.csv — master patient CSV with predictions")
print(" 11. layer_relevance/         — per-layer neuron relevance CSVs")

Computing LRP attributions …


d:\USTHB\M2\Stage Parkinson\Parkinson-prediction\.venv\Lib\site-packages\captum\attr\_core\lrp.py:207: UserWarning: Input Tensor 0 did not already require gradients, required_grads has been set automatically.
  gradient_mask = apply_gradient_requirements(input_tuple)
d:\USTHB\M2\Stage Parkinson\Parkinson-prediction\.venv\Lib\site-packages\captum\attr\_core\layer\layer_lrp.py:233: UserWarning: Input Tensor 0 did not already require gradients, required_grads has been set automatically.
  gradient_mask = apply_gradient_requirements(inputs_tuple)


Overall accuracy: 0.868
  Saved: lrp_outputs_thesis\global_importance/top_global_features_signed.png
  Saved: signed_relevance_heatmap.png
  Saved: clustered_abs_relevance_heatmap.png
  Saved: correct_vs_misclassified_relevance.png
  Saved: relevance_stability_heatmap.png
  Saved: pca_relevance.png
  Saved: tsne_relevance.png


d:\USTHB\M2\Stage Parkinson\Parkinson-prediction\.venv\Lib\site-packages\numpy\lib\function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
d:\USTHB\M2\Stage Parkinson\Parkinson-prediction\.venv\Lib\site-packages\numpy\lib\function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


  Saved: feature_correlation_top_relevant.png
  Saved: variance_vs_relevance.png
  Saved: relevance_flow_across_layers.png


...

  Saved: stage_progression_importance_curve.png
  Saved: per_stage_signed_relevance.png
  Saved: neuron_relevance_distribution.png
  Saved: neuron_heatmap_layer1_linear.png
  Saved: neuron_heatmap_layer2_linear.png
  Saved: neuron_heatmap_layer3_linear.png
  Saved: neuron_heatmap_classifier.png


C:\Users\pc\AppData\Local\Temp\ipykernel_3644\50420688.py:674: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap_layers = plt.cm.get_cmap('Set2', len(layer_names))


  Saved: layer_specialization_entropy.png


C:\Users\pc\AppData\Local\Temp\ipykernel_3644\50420688.py:702: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap_l = plt.cm.get_cmap('tab10', len(layer_names))


  Saved: cross_layer_relevance_propagation.png

✓ Layer relevance plots saved to: lrp_outputs_thesis\layer_relevance\plots

✓ All thesis outputs saved to: lrp_outputs_thesis

Output summary:
  1. global_importance/       — signed global bar chart + CSV
  2. heatmaps/                — signed diverging + clustered heatmaps
  3. correct_vs_misclassified/ — relevance comparison correct/wrong patients
  4. relevance_stability/     — mean + coefficient-of-variation heatmaps
  5. embeddings/              — PCA + t-SNE of relevance vectors
  6. correlation_analysis/    — feature correlation + variance-vs-relevance
  7. relevance_flow/          — layer-by-layer flow curve
  8. stage_progression/       — progression curve top features
  9. class_importance/        — per-stage signed bar charts + CSVs
 10. relevance_per_patient.csv — master patient CSV with predictions
 11. layer_relevance/         — per-layer neuron relevance CSVs
